# Public Portfolio Version — Project Controls Recovery Optimization

> **Confidentiality notice**
>
> This is a sanitized portfolio notebook. Original project files and executed project data are intentionally excluded.
> The public version does **not** include IFC geometry, Primavera P6/XER source files, S10 exports, PBIX files,
> coordinates, real resource identifiers, original activity names, original calendar identifiers or executed outputs.
>
> Activity IDs, dates, labels and sample trend values in this notebook have been anonymized or shifted.

### Architecture demonstrated

**P6 snapshots → EVM / Forecast → CPM network → Recovery candidates → Resource-based recovery cost → 81 scenarios → Exact Pareto frontier → NSGA-II validation → Power BI data mart**

The algorithm explores alternatives; the project team validates assumptions and makes the final decision.



# Project Controls Intelligence — Linear Infrastructure Recovery Demo

**Pipeline reproducible y auditable de Project Controls**

**P6 snapshots → S10 → calidad de datos → costos → snapshots → Power BI → forecast → recovery → IA**

Este notebook será la fuente trazable del proyecto.  
Power BI funcionará como **capa de visualización y decisión**, mientras que aquí quedará documentada la lógica de transformación, validación y cálculo.

### Principios
- No modificar los archivos fuente.
- Mantener trazabilidad por `Activity ID` y `Resource ID`.
- Separar **valor contractual** de **costo interno de producción**.
- Registrar cada control como `PASS`, `WARNING` o `ERROR`.
- Exportar únicamente tablas curadas para Power BI y, posteriormente, para el agente de IA.


## 00 — Configuración del proyecto

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import re
from datetime import datetime

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT = {
    "project_id": "C01",
    "project_name": "LINEAR INFRASTRUCTURE RECOVERY DEMO",
    "currency": "PEN",
    "baseline_finish": "2026-01-08",
}

CUTS = {
    "C01": pd.Timestamp("2025-05-31"),
    "C02": pd.Timestamp("2025-06-30"),
    "C03": pd.Timestamp("2025-07-31"),
    "C04": pd.Timestamp("2025-08-31"),
}

DATA_DIR = Path("/content/data")
OUTPUT_DIR = Path("/content/output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(PROJECT["project_name"])
print("Cortes:", CUTS)



## 01 — Carga de archivos

En Colab, ejecuta esta celda y selecciona los Excel exportados desde P6 y S10.

La convención recomendada es:

- `C01_Mayo_2026_Activities.xlsx`
- `C01_Mayo_2026_Resource_Assignments.xlsx`
- `C02_Junio_2026_Activities.xlsx`
- `C02_Junio_2026_Resource_Assignments.xlsx`
- `C03_Julio_2026_Activities.xlsx`
- `C03_Julio_2026_Resource_Assignments.xlsx`
- `C04_Agosto_2026_Activities.xlsx`
- `C04_Agosto_2026_Resource_Assignments.xlsx`
- `01_Activities.xlsx`  ← programación/base original
- `04_Expenses.xlsx`    ← valor contractual por actividad
- `07_insumo_s10.xlsx`  ← catálogo S10 con PU


In [ ]:

# Ejecutar solamente en Google Colab.
try:
    from google.colab import files
    uploaded = files.upload()

    for name, content in uploaded.items():
        path = DATA_DIR / name
        path.write_bytes(content)

    print(f"Archivos cargados: {len(uploaded)}")
except ImportError:
    print("No estás en Google Colab. Copia los archivos manualmente a:", DATA_DIR)


In [ ]:
DATA_DIR = Path("/content/data")
OUTPUT_DIR = Path("/content/output")

In [ ]:
from pathlib import Path
import shutil

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

archivos_esperados = [
    "01_Activities.xlsx",
    "04_Expenses.xlsx",
    "07_insumo_s10.xlsx",
    "C01_Mayo_2026_Activities.xlsx",
    "C01_Mayo_2026_Resource_Assignments.xlsx",
    "C02_Junio_2026_Activities.xlsx",
    "C02_Junio_2026_Resource_Assignments.xlsx",
    "C03_Julio_2026_Activities.xlsx",
    "C03_Julio_2026_Resource_Assignments.xlsx",
    "C04_Agosto_2026_Activities.xlsx",
    "C04_Agosto_2026_Resource_Assignments.xlsx",
]

for nombre in archivos_esperados:
    origen = Path("/content") / nombre
    destino = DATA_DIR / nombre

    if destino.exists():
        print(f"✅ Ya está en data: {nombre}")
    elif origen.exists():
        shutil.copy2(origen, destino)
        print(f"➡️ Copiado a data: {nombre}")
    else:
        print(f"❌ No encontrado: {nombre}")

## 02 — Manifiesto y trazabilidad SHA-256

In [ ]:

FILE_MANIFEST = {
    "baseline_activities": "01_Activities.xlsx",
    "expenses": "04_Expenses.xlsx",
    "s10": "07_insumo_s10.xlsx",

    "C01_activities": "C01_Mayo_2026_Activities.xlsx",
    "C01_resources": "C01_Mayo_2026_Resource_Assignments.xlsx",

    "C02_activities": "C02_Junio_2026_Activities.xlsx",
    "C02_resources": "C02_Junio_2026_Resource_Assignments.xlsx",

    "C03_activities": "C03_Julio_2026_Activities.xlsx",
    "C03_resources": "C03_Julio_2026_Resource_Assignments.xlsx",

    "C04_activities": "C04_Agosto_2026_Activities.xlsx",
    "C04_resources": "C04_Agosto_2026_Resource_Assignments.xlsx",
}

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for key, filename in FILE_MANIFEST.items():
    path = DATA_DIR / filename
    manifest_rows.append({
        "dataset": key,
        "filename": filename,
        "exists": path.exists(),
        "sha256": sha256_file(path) if path.exists() else None,
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

df_manifest = pd.DataFrame(manifest_rows)
df_manifest



## 03 — Funciones de lectura y normalización

Los exports de P6 contienen una primera fila interna con los nombres visibles de los campos.  
La función siguiente la elimina automáticamente y conserva los nombres técnicos de columnas.


In [ ]:

def read_p6_excel(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, dtype=object)

    # La primera fila de los exports P6 contiene etiquetas como "Activity ID" o "Resource ID".
    if len(df) > 0:
        first = df.iloc[0].astype(str).str.strip().str.lower()
        markers = {
            "activity id", "resource id", "expense item",
            "(*)activity status", "budgeted units(h)"
        }
        if any(v in markers for v in first.values):
            df = df.iloc[1:].copy()

    df = df.reset_index(drop=True)
    return df

def clean_id(series):
    return (
        series.astype("string")
        .str.strip()
        .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def to_num(series):
    return pd.to_numeric(series, errors="coerce")

def to_date(series):
    return pd.to_datetime(series, errors="coerce")

def add_audit(log, control, result, detail, dataset=None):
    log.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "dataset": dataset,
        "control": control,
        "result": result,
        "detail": detail,
    })


## 04 — Ingesta de fuentes

In [ ]:

audit_log = []

# Fuentes maestras
baseline_path = DATA_DIR / FILE_MANIFEST["baseline_activities"]
expenses_path = DATA_DIR / FILE_MANIFEST["expenses"]
s10_path = DATA_DIR / FILE_MANIFEST["s10"]

baseline = read_p6_excel(baseline_path, "TASK") if baseline_path.exists() else pd.DataFrame()
expenses = read_p6_excel(expenses_path, "PROJCOST") if expenses_path.exists() else pd.DataFrame()
s10 = pd.read_excel(s10_path, sheet_name="Insumos_S10", dtype={"CodInsumo": str}) if s10_path.exists() else pd.DataFrame()

activity_snapshots = {}
resource_snapshots = {}

for cut in CUTS:
    a_path = DATA_DIR / FILE_MANIFEST[f"{cut}_activities"]
    r_path = DATA_DIR / FILE_MANIFEST[f"{cut}_resources"]

    if a_path.exists():
        activity_snapshots[cut] = read_p6_excel(a_path, "TASK")
    if r_path.exists():
        resource_snapshots[cut] = read_p6_excel(r_path, "TASKRSRC")

print("Snapshots Activities:", list(activity_snapshots))
print("Snapshots Resources:", list(resource_snapshots))
print("S10:", s10.shape)


## 05 — Estandarización de snapshots

In [ ]:

ACT_NUMERIC = [
    "calc_phys_complete_pct", "target_drtn_hr_cnt", "remain_drtn_hr_cnt",
    "total_float_hr_cnt", "target_cost", "act_cost", "remain_cost",
    "target_work_qty", "act_work_qty", "remain_work_qty",
]
ACT_DATES = ["act_start_date", "act_end_date", "start_date", "end_date"]

RSRC_NUMERIC = [
    "target_qty", "act_qty", "remain_qty", "total_qty",
    "cost_per_qty", "target_cost", "act_cost", "remain_cost", "total_cost",
]
RSRC_DATES = ["start_date", "end_date"]

for cut, df in activity_snapshots.items():
    df["task_code"] = clean_id(df["task_code"])
    for c in ACT_NUMERIC:
        if c in df.columns:
            df[c] = to_num(df[c])
    for c in ACT_DATES:
        if c in df.columns:
            df[c] = to_date(df[c])

    df["Cut_ID"] = cut
    df["Snapshot_Date"] = CUTS[cut]
    activity_snapshots[cut] = df

for cut, df in resource_snapshots.items():
    df["rsrc_id"] = clean_id(df["rsrc_id"])
    df["task_id"] = clean_id(df["task_id"])
    for c in RSRC_NUMERIC:
        if c in df.columns:
            df[c] = to_num(df[c])
    for c in RSRC_DATES:
        if c in df.columns:
            df[c] = to_date(df[c])

    df["Cut_ID"] = cut
    df["Snapshot_Date"] = CUTS[cut]
    resource_snapshots[cut] = df

if not s10.empty:
    s10["CodInsumo"] = clean_id(s10["CodInsumo"])
    s10["Precio_unit"] = to_num(s10["Precio_unit"])

print("Estandarización completada.")


## 06 — Data Quality: controles de gobernanza

In [ ]:

# 1. Archivos requeridos
for _, row in df_manifest.iterrows():
    result = "PASS" if row["exists"] else "ERROR"
    add_audit(
        audit_log,
        "FILE_EXISTS",
        result,
        f'{row["filename"]}: {"disponible" if row["exists"] else "faltante"}',
        row["dataset"],
    )

# 2. Duplicados en catálogo S10
if not s10.empty:
    dup_s10 = s10["CodInsumo"].duplicated(keep=False).sum()
    add_audit(
        audit_log,
        "S10_RESOURCE_ID_UNIQUE",
        "PASS" if dup_s10 == 0 else "ERROR",
        f"Resource ID duplicados en S10: {dup_s10}",
        "s10",
    )

# 3. Controles por snapshot de actividades
for cut, df in activity_snapshots.items():
    productive = df[df["task_code"].astype(str).str.match(r"^A\d+$", na=False)].copy()

    dup = productive["task_code"].duplicated().sum()
    add_audit(audit_log, "ACTIVITY_ID_UNIQUE",
              "PASS" if dup == 0 else "ERROR",
              f"Duplicados: {dup}", cut)

    invalid_pct = (~productive["calc_phys_complete_pct"].between(0, 100, inclusive="both")).sum()
    add_audit(audit_log, "PHYSICAL_PERCENT_RANGE",
              "PASS" if invalid_pct == 0 else "ERROR",
              f"Fuera de 0-100%: {invalid_pct}", cut)

    completed = productive["status_code"].astype(str).str.lower().eq("completed")
    no_finish = (completed & productive["act_end_date"].isna()).sum()
    add_audit(audit_log, "COMPLETED_HAS_ACTUAL_FINISH",
              "PASS" if no_finish == 0 else "ERROR",
              f"Completadas sin Actual Finish: {no_finish}", cut)

    neg_rd = (productive["remain_drtn_hr_cnt"].fillna(0) < 0).sum()
    add_audit(audit_log, "NO_NEGATIVE_REMAINING_DURATION",
              "PASS" if neg_rd == 0 else "ERROR",
              f"RD negativas: {neg_rd}", cut)

# 4. Cobertura P6 ↔ S10
if not s10.empty:
    s10_ids = set(s10["CodInsumo"].dropna())
    for cut, df in resource_snapshots.items():
        p6_ids = set(df["rsrc_id"].dropna())
        missing = sorted(p6_ids - s10_ids)
        coverage = 100 * (len(p6_ids) - len(missing)) / len(p6_ids) if p6_ids else np.nan

        add_audit(
            audit_log,
            "P6_S10_RESOURCE_COVERAGE",
            "PASS" if len(missing) == 0 else "ERROR",
            f"{len(p6_ids)-len(missing)}/{len(p6_ids)} = {coverage:.1f}% | faltantes: {missing[:10]}",
            cut,
        )

df_audit = pd.DataFrame(audit_log)
df_audit



## 07 — Fact Activity Snapshot

Una fila por **Actividad × Corte**.  
Esta será la tabla principal para analizar evolución de avance, fechas, float y forecast.


In [ ]:

fact_activity_snapshot = pd.concat(
    activity_snapshots.values(),
    ignore_index=True
) if activity_snapshots else pd.DataFrame()

if not fact_activity_snapshot.empty:
    fact_activity_snapshot = fact_activity_snapshot[
        fact_activity_snapshot["task_code"].astype(str).str.match(r"^A\d+$", na=False)
    ].copy()

    fact_activity_snapshot["Physical_Pct"] = fact_activity_snapshot["calc_phys_complete_pct"] / 100.0
    fact_activity_snapshot["Forecast_Finish"] = fact_activity_snapshot["end_date"]
    fact_activity_snapshot["Actual_Start"] = fact_activity_snapshot["act_start_date"]
    fact_activity_snapshot["Actual_Finish"] = fact_activity_snapshot["act_end_date"]

fact_activity_snapshot.head()



## 08 — Fact Resource Snapshot + integración S10

Aquí se reconstruye el **costo interno de producción**:

- `Budget_Cost_Internal = Budgeted Units × PU S10`
- `Actual_Cost_Internal = Actual Units × PU S10`
- `Remaining_Cost_Internal = Remaining Units × PU S10`
- `EAC_Cost_Internal = At Completion Units × PU S10`


In [ ]:

fact_resource_snapshot = pd.concat(
    resource_snapshots.values(),
    ignore_index=True
) if resource_snapshots else pd.DataFrame()

if not fact_resource_snapshot.empty and not s10.empty:
    dim_resource = (
        s10[["CodInsumo", "Recurso", "Unidad", "Precio_unit"]]
        .drop_duplicates("CodInsumo")
        .rename(columns={"CodInsumo": "Resource_ID"})
        .copy()
    )

    fact_resource_snapshot = fact_resource_snapshot.rename(columns={
        "rsrc_id": "Resource_ID",
        "task_id": "Activity_ID",
    })

    fact_resource_snapshot = fact_resource_snapshot.merge(
        dim_resource,
        how="left",
        on="Resource_ID",
        validate="many_to_one",
    )

    fact_resource_snapshot["Budget_Cost_Internal"] = fact_resource_snapshot["target_qty"] * fact_resource_snapshot["Precio_unit"]
    fact_resource_snapshot["Actual_Cost_Internal"] = fact_resource_snapshot["act_qty"] * fact_resource_snapshot["Precio_unit"]
    fact_resource_snapshot["Remaining_Cost_Internal"] = fact_resource_snapshot["remain_qty"] * fact_resource_snapshot["Precio_unit"]
    fact_resource_snapshot["EAC_Cost_Internal"] = fact_resource_snapshot["total_qty"] * fact_resource_snapshot["Precio_unit"]

fact_resource_snapshot.head()


In [ ]:
print("Filas Fact Activity:", len(fact_activity_snapshot))
print("Actividades únicas:", fact_activity_snapshot["task_code"].nunique())
print()

print("Filas Fact Resource:", len(fact_resource_snapshot))
print("Recursos únicos:", fact_resource_snapshot["Resource_ID"].nunique())
print("PU S10 faltantes:", fact_resource_snapshot["Precio_unit"].isna().sum())

fact_resource_snapshot[
    [
        "Cut_ID",
        "Activity_ID",
        "Resource_ID",
        "Recurso",
        "act_qty",
        "Precio_unit",
        "Actual_Cost_Internal"
    ]
].head(10)

## 09 — Dimensiones maestras

In [ ]:

if not baseline.empty:
    baseline["task_code"] = clean_id(baseline["task_code"])

    dim_activity = baseline[
        baseline["task_code"].astype(str).str.match(r"^A\d+$", na=False)
    ][["task_code", "wbs_id", "task_name"]].drop_duplicates("task_code").rename(columns={
        "task_code": "Activity_ID",
        "wbs_id": "WBS_ID",
        "task_name": "Activity_Name",
    })
else:
    cols = ["task_code", "wbs_id", "task_name"]
    dim_activity = (
        fact_activity_snapshot[cols]
        .drop_duplicates("task_code")
        .rename(columns={
            "task_code": "Activity_ID",
            "wbs_id": "WBS_ID",
            "task_name": "Activity_Name",
        })
        if not fact_activity_snapshot.empty else pd.DataFrame()
    )

dim_activity.head()


## 10 — Resumen preliminar por corte

In [ ]:

summary_rows = []

for cut in sorted(resource_snapshots):
    r = fact_resource_snapshot[fact_resource_snapshot["Cut_ID"] == cut]
    a = fact_activity_snapshot[fact_activity_snapshot["Cut_ID"] == cut]

    summary_rows.append({
        "Cut_ID": cut,
        "Snapshot_Date": CUTS[cut],
        "Activities": a["task_code"].nunique(),
        "Physical_Avg_Unweighted": a["Physical_Pct"].mean(),
        "Budget_Cost_Internal": r["Budget_Cost_Internal"].sum(),
        "Actual_Cost_Internal": r["Actual_Cost_Internal"].sum(),
        "Remaining_Cost_Internal": r["Remaining_Cost_Internal"].sum(),
        "EAC_Cost_Internal": r["EAC_Cost_Internal"].sum(),
    })

df_cut_summary = pd.DataFrame(summary_rows)

if not df_cut_summary.empty:
    df_cut_summary["CPI_Proxy"] = (
        df_cut_summary["Budget_Cost_Internal"] * df_cut_summary["Physical_Avg_Unweighted"]
    ) / df_cut_summary["Actual_Cost_Internal"].replace(0, np.nan)

df_cut_summary


In [ ]:
resumen = df_cut_summary.copy()

cols_money = [
    "Budget_Cost_Internal",
    "Actual_Cost_Internal",
    "Remaining_Cost_Internal",
    "EAC_Cost_Internal"
]

for c in cols_money:
    resumen[c] = resumen[c].round(2)

resumen[
    [
        "Cut_ID",
        "Snapshot_Date",
        "Budget_Cost_Internal",
        "Actual_Cost_Internal",
        "Remaining_Cost_Internal",
        "EAC_Cost_Internal"
    ]
]

In [ ]:
fact_resource_snapshot.groupby("Cut_ID").agg(
    Budget_Internal=("Budget_Cost_Internal", "sum"),
    Actual_Internal=("Actual_Cost_Internal", "sum"),
    Remaining_Internal=("Remaining_Cost_Internal", "sum"),
    EAC_Internal=("EAC_Cost_Internal", "sum")
).round(2)


> **Nota:** `CPI_Proxy` todavía no es el CPI definitivo.  
> El EVM profesional se calculará después usando el presupuesto interno por actividad y el avance físico ponderado, no un promedio simple.


## 11 — Exportación de tablas curadas para Power BI

In [ ]:

outputs = {
    "dim_activity.csv": dim_activity,
    "dim_resource.csv": dim_resource if "dim_resource" in globals() else pd.DataFrame(),
    "fact_activity_snapshot.csv": fact_activity_snapshot,
    "fact_resource_snapshot.csv": fact_resource_snapshot,
    "audit_log.csv": df_audit,
    "file_manifest.csv": df_manifest,
    "cut_summary_preliminary.csv": df_cut_summary,
}

for filename, df in outputs.items():
    df.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p.name)


## 12 — Descargar paquete curado

In [ ]:

import shutil

zip_base = "/content/Linear_Infrastructure_Demo_Project_Controls_Curated"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print(zip_base + ".zip")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except ImportError:
    pass


In [ ]:
## 13 — Earned Value Management: EV, AC, CPI y CV

In [ ]:
# ============================================================
# 13 — EVM: EV, AC, CPI y CV
# ============================================================

# 1. Presupuesto interno por actividad
# Debe ser estable entre cortes, porque proviene de Budgeted Units × PU S10

budget_internal_check = (
    fact_resource_snapshot
    .groupby(["Cut_ID", "Activity_ID"], as_index=False)
    .agg(
        Budget_Cost_Internal=("Budget_Cost_Internal", "sum"),
        Actual_Cost_Internal=("Actual_Cost_Internal", "sum"),
        Remaining_Cost_Internal=("Remaining_Cost_Internal", "sum"),
        EAC_Cost_Internal=("EAC_Cost_Internal", "sum"),
    )
)

# Validar que el Budget interno de cada actividad no cambie entre cortes
budget_variation = (
    budget_internal_check
    .groupby("Activity_ID")["Budget_Cost_Internal"]
    .agg(["min", "max"])
)

budget_variation["diff"] = (
    budget_variation["max"] - budget_variation["min"]
)

print(
    "Actividades con variación de Budget interno entre cortes:",
    (budget_variation["diff"].abs() > 0.01).sum()
)

# 2. Tomar el Budget interno único por actividad
budget_internal_activity = (
    budget_internal_check
    .groupby("Activity_ID", as_index=False)
    .agg(
        Budget_Cost_Internal=("Budget_Cost_Internal", "first")
    )
)

# 3. Costos reales / forecast por actividad y corte
cost_by_activity_cut = (
    budget_internal_check[
        [
            "Cut_ID",
            "Activity_ID",
            "Actual_Cost_Internal",
            "Remaining_Cost_Internal",
            "EAC_Cost_Internal"
        ]
    ]
    .copy()
)

# 4. Base EVM desde Fact Activity Snapshot
fact_evm = fact_activity_snapshot[
    [
        "Cut_ID",
        "Snapshot_Date",
        "task_code",
        "task_name",
        "Physical_Pct",
        "target_cost",
        "Forecast_Finish",
        "Actual_Start",
        "Actual_Finish",
        "total_float_hr_cnt",
        "remain_drtn_hr_cnt"
    ]
].copy()

fact_evm = fact_evm.rename(columns={
    "task_code": "Activity_ID",
    "task_name": "Activity_Name",
    "target_cost": "Budget_Contractual"
})

# 5. Integrar presupuesto interno
fact_evm = fact_evm.merge(
    budget_internal_activity,
    how="left",
    on="Activity_ID",
    validate="many_to_one"
)

# 6. Integrar costos reales internos por corte
fact_evm = fact_evm.merge(
    cost_by_activity_cut,
    how="left",
    on=["Cut_ID", "Activity_ID"],
    validate="one_to_one"
)

# 7. Earned Value CONTRACTUAL
fact_evm["EV_Contractual"] = (
    fact_evm["Budget_Contractual"]
    * fact_evm["Physical_Pct"]
)

# 8. Earned Value INTERNO
fact_evm["EV_Internal"] = (
    fact_evm["Budget_Cost_Internal"]
    * fact_evm["Physical_Pct"]
)

# 9. Actual Cost interno
fact_evm["AC_Internal"] = fact_evm["Actual_Cost_Internal"]

# 10. CPI interno
fact_evm["CPI_Internal"] = (
    fact_evm["EV_Internal"]
    / fact_evm["AC_Internal"].replace(0, np.nan)
)

# 11. Cost Variance interno
fact_evm["CV_Internal"] = (
    fact_evm["EV_Internal"]
    - fact_evm["AC_Internal"]
)

# 12. Variance at Completion según forecast de recursos
fact_evm["VAC_Internal"] = (
    fact_evm["Budget_Cost_Internal"]
    - fact_evm["EAC_Cost_Internal"]
)

# 13. Sobrecosto proyectado
fact_evm["Forecast_Cost_Variance"] = (
    fact_evm["EAC_Cost_Internal"]
    - fact_evm["Budget_Cost_Internal"]
)

fact_evm.head()

In [ ]:
evm_summary = (
    fact_evm
    .groupby(["Cut_ID", "Snapshot_Date"], as_index=False)
    .agg(
        BAC_Contractual=("Budget_Contractual", "sum"),
        BAC_Internal=("Budget_Cost_Internal", "sum"),
        EV_Contractual=("EV_Contractual", "sum"),
        EV_Internal=("EV_Internal", "sum"),
        AC_Internal=("AC_Internal", "sum"),
        EAC_Internal=("EAC_Cost_Internal", "sum"),
    )
)

evm_summary["CPI_Internal"] = (
    evm_summary["EV_Internal"]
    / evm_summary["AC_Internal"].replace(0, np.nan)
)

evm_summary["CV_Internal"] = (
    evm_summary["EV_Internal"]
    - evm_summary["AC_Internal"]
)

evm_summary["VAC_Internal"] = (
    evm_summary["BAC_Internal"]
    - evm_summary["EAC_Internal"]
)

evm_summary.round(2)


---

# Próximos módulos

Una vez validada esta base:

### 06 — Earned Value Management
- PV contractual
- EV contractual
- EV interno
- AC interno
- SPI
- CPI
- SV / CV
- EAC / ETC / VAC

### 07 — Trend Register
- Tendencias incorporadas
- Open Trends
- Impacto plazo/costo
- Current Forecast vs Potential Forecast

### 08 — Forecast & Recovery
- Baseline Finish
- Current Forecast
- Recovery scenarios

### 09 — NSGA-II
- Días recuperados
- Costo incremental
- Recursos adicionales
- Pareto frontier
- Escenario recomendado

### 10 — AI Agent
El agente consultará **tablas curadas y herramientas verificables**, no los Excel crudos.


In [ ]:
# Inspección segura de la baseline

print("Columnas disponibles en baseline:")
print(baseline.columns.tolist())

display(
    baseline[
        baseline["task_code"].astype(str).str.match(r"^A\d+$", na=False)
    ][
        [
            "task_code",
            "task_name",
            "start_date",
            "end_date"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# 14 — Baseline de actividades para Planned Value
# ============================================================

baseline_plan = (
    baseline[
        baseline["task_code"].astype(str).str.match(r"^A\d+$", na=False)
    ][
        [
            "task_code",
            "task_name",
            "start_date",
            "end_date"
        ]
    ]
    .copy()
)

baseline_plan = baseline_plan.rename(columns={
    "task_code": "Activity_ID",
    "task_name": "Activity_Name_Baseline",
    "start_date": "BL_Start",
    "end_date": "BL_Finish"
})

baseline_plan["BL_Start"] = pd.to_datetime(
    baseline_plan["BL_Start"],
    errors="coerce"
)

baseline_plan["BL_Finish"] = pd.to_datetime(
    baseline_plan["BL_Finish"],
    errors="coerce"
)

print("Actividades baseline:", len(baseline_plan))
print("Fechas BL Start faltantes:", baseline_plan["BL_Start"].isna().sum())
print("Fechas BL Finish faltantes:", baseline_plan["BL_Finish"].isna().sum())

display(baseline_plan.head(10))

In [ ]:
budget_contractual_activity = (
    fact_evm[
        ["Activity_ID", "Budget_Contractual"]
    ]
    .drop_duplicates("Activity_ID")
)

baseline_plan = baseline_plan.merge(
    budget_contractual_activity,
    how="left",
    on="Activity_ID",
    validate="one_to_one"
)

print(
    "Actividades sin presupuesto contractual:",
    baseline_plan["Budget_Contractual"].isna().sum()
)

print(
    "BAC contractual reconstruido:",
    baseline_plan["Budget_Contractual"].sum()
)

## 15 — Planned Value (PV) y Schedule Performance

In [ ]:
# ============================================================
# 15 — Reconstrucción del Planned Value
# Método: distribución lineal del presupuesto entre
# BL Start y BL Finish de cada actividad.
# ============================================================

# Incorporar presupuesto interno a la baseline
baseline_pv = baseline_plan.merge(
    budget_internal_activity,
    how="left",
    on="Activity_ID",
    validate="one_to_one"
)

print(
    "Actividades sin Budget interno:",
    baseline_pv["Budget_Cost_Internal"].isna().sum()
)

# Tabla de cortes
cuts_df = pd.DataFrame({
    "Cut_ID": list(CUTS.keys()),
    "Snapshot_Date": list(CUTS.values())
})

# Cross join: Actividad × Corte
pv_detail = (
    baseline_pv.assign(_key=1)
    .merge(cuts_df.assign(_key=1), on="_key")
    .drop(columns="_key")
)

# Duración baseline en días calendario
pv_detail["BL_Duration_Days"] = (
    pv_detail["BL_Finish"] - pv_detail["BL_Start"]
).dt.total_seconds() / 86400

pv_detail["Elapsed_Days"] = (
    pv_detail["Snapshot_Date"] - pv_detail["BL_Start"]
).dt.total_seconds() / 86400

# % planificado reconstruido
before_start = pv_detail["Snapshot_Date"] <= pv_detail["BL_Start"]
after_finish = pv_detail["Snapshot_Date"] >= pv_detail["BL_Finish"]
zero_duration = pv_detail["BL_Duration_Days"] <= 0

pv_detail["Planned_Pct"] = np.where(
    before_start,
    0.0,
    np.where(
        after_finish,
        1.0,
        np.where(
            zero_duration,
            1.0,
            pv_detail["Elapsed_Days"] / pv_detail["BL_Duration_Days"]
        )
    )
)

pv_detail["Planned_Pct"] = pv_detail["Planned_Pct"].clip(0, 1)

# Planned Value contractual
pv_detail["PV_Contractual"] = (
    pv_detail["Budget_Contractual"]
    * pv_detail["Planned_Pct"]
)

# Planned Value interno
pv_detail["PV_Internal"] = (
    pv_detail["Budget_Cost_Internal"]
    * pv_detail["Planned_Pct"]
)

display(
    pv_detail[
        [
            "Cut_ID",
            "Snapshot_Date",
            "Activity_ID",
            "BL_Start",
            "BL_Finish",
            "Planned_Pct",
            "PV_Contractual",
            "PV_Internal"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# 16 — Resumen integrado EVM
# ============================================================

pv_summary = (
    pv_detail
    .groupby(["Cut_ID", "Snapshot_Date"], as_index=False)
    .agg(
        PV_Contractual=("PV_Contractual", "sum"),
        PV_Internal=("PV_Internal", "sum"),
    )
)

evm_full_summary = evm_summary.merge(
    pv_summary,
    how="left",
    on=["Cut_ID", "Snapshot_Date"],
    validate="one_to_one"
)

# Schedule Performance
evm_full_summary["SPI_Contractual"] = (
    evm_full_summary["EV_Contractual"]
    / evm_full_summary["PV_Contractual"].replace(0, np.nan)
)

evm_full_summary["SPI_Internal"] = (
    evm_full_summary["EV_Internal"]
    / evm_full_summary["PV_Internal"].replace(0, np.nan)
)

# Schedule Variance
evm_full_summary["SV_Contractual"] = (
    evm_full_summary["EV_Contractual"]
    - evm_full_summary["PV_Contractual"]
)

evm_full_summary["SV_Internal"] = (
    evm_full_summary["EV_Internal"]
    - evm_full_summary["PV_Internal"]
)

# Ordenar columnas para lectura ejecutiva
evm_full_summary = evm_full_summary[
    [
        "Cut_ID",
        "Snapshot_Date",
        "BAC_Contractual",
        "BAC_Internal",
        "PV_Contractual",
        "EV_Contractual",
        "PV_Internal",
        "EV_Internal",
        "AC_Internal",
        "SPI_Contractual",
        "SPI_Internal",
        "CPI_Internal",
        "SV_Contractual",
        "SV_Internal",
        "CV_Internal",
        "EAC_Internal",
        "VAC_Internal",
    ]
]

display(evm_full_summary.round(3))

In [ ]:
add_audit(
    audit_log,
    "PV_RECONSTRUCTION_METHOD",
    "WARNING",
    (
        "PV reconstruido mediante distribución lineal por actividad "
        "entre BL Start y BL Finish. No corresponde al time-phased "
        "budget nativo de Primavera P6."
    ),
    "EVM"
)

df_audit = pd.DataFrame(audit_log)

print("Método de PV registrado en Audit Log.")

## 17 — Schedule Forecast y variación de terminación

In [ ]:
# ============================================================
# 17 — Forecast de terminación del proyecto por corte
# Fuente: milestone MILESTONE_FINISH de cada snapshot P6
# ============================================================

schedule_rows = []

baseline_finish = pd.Timestamp(PROJECT["baseline_finish"])

for cut, df in activity_snapshots.items():

    milestone = df[
        df["task_code"].astype(str).str.strip().eq("MILESTONE_FINISH")
    ].copy()

    if len(milestone) == 0:
        forecast_finish = pd.NaT
    else:
        forecast_finish = pd.to_datetime(
            milestone.iloc[0]["end_date"],
            errors="coerce"
        )

    schedule_rows.append({
        "Cut_ID": cut,
        "Snapshot_Date": CUTS[cut],
        "Baseline_Finish": baseline_finish,
        "Forecast_Finish": forecast_finish
    })

schedule_summary = pd.DataFrame(schedule_rows)

schedule_summary["Finish_Variance_Days"] = (
    schedule_summary["Forecast_Finish"].dt.normalize()
    - schedule_summary["Baseline_Finish"].dt.normalize()
).dt.days

display(schedule_summary)

In [ ]:
# ============================================================
# 18 — Executive Project Controls Scorecard
# ============================================================

project_controls_scorecard = evm_full_summary.merge(
    schedule_summary[
        [
            "Cut_ID",
            "Snapshot_Date",
            "Baseline_Finish",
            "Forecast_Finish",
            "Finish_Variance_Days"
        ]
    ],
    how="left",
    on=["Cut_ID", "Snapshot_Date"],
    validate="one_to_one"
)

# Indicadores adicionales
project_controls_scorecard["Cost_Overrun_Forecast"] = (
    project_controls_scorecard["EAC_Internal"]
    - project_controls_scorecard["BAC_Internal"]
)

project_controls_scorecard["Cost_Overrun_Pct"] = (
    project_controls_scorecard["Cost_Overrun_Forecast"]
    / project_controls_scorecard["BAC_Internal"]
)

# Estado simple de gestión
def performance_status(row):

    if (
        row["SPI_Internal"] >= 0.98
        and row["CPI_Internal"] >= 0.98
        and row["Finish_Variance_Days"] <= 3
    ):
        return "CONTROLLED"

    elif (
        row["SPI_Internal"] >= 0.95
        and row["CPI_Internal"] >= 0.95
        and row["Finish_Variance_Days"] <= 15
    ):
        return "WATCH"

    return "CRITICAL"


project_controls_scorecard["Management_Status"] = (
    project_controls_scorecard.apply(
        performance_status,
        axis=1
    )
)

display(
    project_controls_scorecard[
        [
            "Cut_ID",
            "Snapshot_Date",
            "SPI_Internal",
            "CPI_Internal",
            "Baseline_Finish",
            "Forecast_Finish",
            "Finish_Variance_Days",
            "BAC_Internal",
            "EAC_Internal",
            "Cost_Overrun_Forecast",
            "Cost_Overrun_Pct",
            "Management_Status"
        ]
    ].round(3)
)

## 19 — Schedule Driver Analysis

In [ ]:
# ============================================================
# 19 — Schedule Driver Analysis
# Separa drivers activos de impactos aguas abajo
# ============================================================

# 1. PV / baseline por actividad y corte
pv_activity = pv_detail[
    [
        "Cut_ID",
        "Activity_ID",
        "BL_Start",
        "BL_Finish",
        "Planned_Pct",
        "PV_Contractual",
        "PV_Internal"
    ]
].copy()

# 2. Integrar con EVM por actividad
driver_detail = fact_evm.merge(
    pv_activity,
    how="left",
    on=["Cut_ID", "Activity_ID"],
    validate="one_to_one"
)

# 3. Gap de avance
# Positivo = debería haber avanzado más según baseline
driver_detail["Schedule_Gap_Pct"] = (
    driver_detail["Planned_Pct"]
    - driver_detail["Physical_Pct"]
)

# 4. Variación de fecha de término por actividad
driver_detail["Activity_Finish_Variance_Days"] = (
    driver_detail["Forecast_Finish"].dt.normalize()
    - driver_detail["BL_Finish"].dt.normalize()
).dt.days

# 5. Schedule Variance monetaria por actividad
driver_detail["SV_Internal_Activity"] = (
    driver_detail["EV_Internal"]
    - driver_detail["PV_Internal"]
)

# 6. Clasificar estado de ejecución
driver_detail["Execution_Status"] = np.select(
    [
        driver_detail["Physical_Pct"] >= 1,
        driver_detail["Physical_Pct"] > 0
    ],
    [
        "COMPLETED",
        "IN PROGRESS"
    ],
    default="NOT STARTED"
)

# 7. Flag de criticidad
# No interpretamos todavía la unidad del Total Float;
# solamente identificamos actividades con float <= 0.
driver_detail["Critical_Flag"] = (
    driver_detail["total_float_hr_cnt"]
    .fillna(np.inf)
    .le(0)
)

# ============================================================
# C04 — ACTIVIDADES EN EJECUCIÓN
# ============================================================

active_drivers_c04 = (
    driver_detail[
        (driver_detail["Cut_ID"] == "C04")
        &
        (driver_detail["Execution_Status"] == "IN PROGRESS")
    ]
    .copy()
)

active_drivers_c04 = active_drivers_c04.sort_values(
    by=[
        "Critical_Flag",
        "Activity_Finish_Variance_Days",
        "Schedule_Gap_Pct"
    ],
    ascending=[False, False, False]
)

print("=== ACTIVE SCHEDULE DRIVERS — C04 ===")

display(
    active_drivers_c04[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Planned_Pct",
            "Schedule_Gap_Pct",
            "BL_Finish",
            "Forecast_Finish",
            "Activity_Finish_Variance_Days",
            "total_float_hr_cnt",
            "Critical_Flag",
            "SV_Internal_Activity"
        ]
    ].round(3)
)

# ============================================================
# C04 — ACTIVIDADES NO INICIADAS YA DESPLAZADAS
# Estas son exposición downstream, no necesariamente causa raíz.
# ============================================================

downstream_exposure_c04 = (
    driver_detail[
        (driver_detail["Cut_ID"] == "C04")
        &
        (driver_detail["Execution_Status"] == "NOT STARTED")
        &
        (driver_detail["Activity_Finish_Variance_Days"] > 0)
    ]
    .copy()
    .sort_values(
        by="Activity_Finish_Variance_Days",
        ascending=False
    )
)

print()
print("=== DOWNSTREAM SCHEDULE EXPOSURE — C04 ===")

display(
    downstream_exposure_c04[
        [
            "Activity_ID",
            "Activity_Name",
            "BL_Start",
            "BL_Finish",
            "Forecast_Finish",
            "Activity_Finish_Variance_Days",
            "total_float_hr_cnt",
            "Critical_Flag"
        ]
    ].head(15)
)

# ============================================================
# 20 — Trend Register (PUBLIC DEMO)
# Synthetic examples used to demonstrate the data model.
# ============================================================

trend_register = pd.DataFrame([
    {
        "Trend_ID": "TR-001",
        "Activity_ID": "RC01",
        "Trend_Name": "Lower-than-planned productivity",
        "Category": "PRODUCTIVITY",
        "Type": "NEGATIVE",
        "Status": "INCORPORATED",
        "Detected_Cut": "C03",
        "Schedule_Impact_Days": 4,
        "Cost_Impact": 5000,
        "Probability": 1.00,
        "Description": "Synthetic portfolio example: productivity below the planning assumption."
    },
    {
        "Trend_ID": "TR-002",
        "Activity_ID": "RC02",
        "Trend_Name": "Resource availability constraint",
        "Category": "RESOURCE",
        "Type": "NEGATIVE",
        "Status": "OPEN",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 2,
        "Cost_Impact": 3000,
        "Probability": 0.60,
        "Description": "Synthetic portfolio example: possible limitation in resource availability."
    },
    {
        "Trend_ID": "TR-003",
        "Activity_ID": "ACT05",
        "Trend_Name": "Positive productivity trend",
        "Category": "PRODUCTIVITY",
        "Type": "POSITIVE",
        "Status": "INCORPORATED",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": -1,
        "Cost_Impact": -1000,
        "Probability": 0.80,
        "Description": "Synthetic portfolio example: improved productivity versus the planning assumption."
    }
])

display(trend_register)


In [ ]:
# ============================================================
# 20 — Trend Register
# Tendencias identificadas durante la ejecución
# ============================================================

trend_register = pd.DataFrame([
    {
        "Trend_ID": "TR-001",
        "Activity_ID": "RC01",
        "Trend_Name": "Menor productividad en mejoramiento de suelo",
        "Category": "PRODUCTIVITY",
        "Type": "NEGATIVE",
        "Status": "INCORPORATED",
        "Detected_Cut": "C03",
        "Schedule_Impact_Days": 6,
        "Cost_Impact": 18000,
        "Probability": 1.00,
        "Description":
            "La productividad observada en RC01 resulta inferior a la prevista, "
            "generando pérdida progresiva de avance y afectación de la ruta crítica."
    },

    {
        "Trend_ID": "TR-002",
        "Activity_ID": "RC01",
        "Trend_Name": "Persistencia de baja productividad en agosto",
        "Category": "PRODUCTIVITY",
        "Type": "NEGATIVE",
        "Status": "INCORPORATED",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 5,
        "Cost_Impact": 12000,
        "Probability": 1.00,
        "Description":
            "La desviación de productividad continúa durante agosto, "
            "incrementando el forecast final del proyecto de +6 a +11 días."
    },

    {
        "Trend_ID": "TR-003",
        "Activity_ID": "ACT05",
        "Trend_Name": "Buen desempeño en instalación de acero trapezoidal",
        "Category": "PRODUCTIVITY",
        "Type": "POSITIVE",
        "Status": "INCORPORATED",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 0,
        "Cost_Impact": -2500,
        "Probability": 1.00,
        "Description":
            "La actividad mantiene su fecha baseline y conserva holgura, "
            "evitando una desviación adicional del frente trapezoidal."
    },

    {
        "Trend_ID": "TR-004",
        "Activity_ID": "RC02",
        "Trend_Name": "Riesgo de baja disponibilidad inicial de encofrado",
        "Category": "RESOURCE",
        "Type": "NEGATIVE",
        "Status": "OPEN",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 3,
        "Cost_Impact": 6000,
        "Probability": 0.60,
        "Description":
            "La disponibilidad de encofrado podría limitar el arranque "
            "del frente rectangular una vez liberado."
    },

    {
        "Trend_ID": "TR-005",
        "Activity_ID": "RC03",
        "Trend_Name": "Curva de aprendizaje en concreto estructural",
        "Category": "PRODUCTIVITY",
        "Type": "POSITIVE",
        "Status": "OPEN",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": -2,
        "Cost_Impact": -4000,
        "Probability": 0.50,
        "Description":
            "La repetitividad del proceso puede mejorar la productividad "
            "del concreto en el frente rectangular."
    },

    {
        "Trend_ID": "TR-006",
        "Activity_ID": "ACT06",
        "Trend_Name": "Restricción temporal de frente de trabajo",
        "Category": "CONSTRAINT",
        "Type": "NEGATIVE",
        "Status": "OPEN",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 4,
        "Cost_Impact": 8500,
        "Probability": 0.40,
        "Description":
            "Existe exposición a una restricción temporal del frente "
            "durante los trabajos de muros."
    },

    {
        "Trend_ID": "TR-007",
        "Activity_ID": "RC01",
        "Trend_Name": "Refuerzo dirigido de cuadrilla",
        "Category": "RECOVERY",
        "Type": "POSITIVE",
        "Status": "PROPOSED",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": -3,
        "Cost_Impact": 9500,
        "Probability": 0.70,
        "Description":
            "El refuerzo selectivo de recursos podría recuperar parte "
            "de la desviación de RC01."
    },

    {
        "Trend_ID": "TR-008",
        "Activity_ID": "ACT07",
        "Trend_Name": "Riesgo de productividad en acabados",
        "Category": "PRODUCTIVITY",
        "Type": "NEGATIVE",
        "Status": "OPEN",
        "Detected_Cut": "C04",
        "Schedule_Impact_Days": 3,
        "Cost_Impact": 5000,
        "Probability": 0.35,
        "Description":
            "El rendimiento de acabados podría generar presión adicional "
            "sobre las actividades finales del proyecto."
    }
])

display(trend_register)

In [ ]:
# ============================================================
# 20.1 — Open Trends Exposure
# ============================================================

open_trends = trend_register[
    trend_register["Status"] == "OPEN"
].copy()

open_trends["Expected_Schedule_Impact"] = (
    open_trends["Schedule_Impact_Days"]
    * open_trends["Probability"]
)

open_trends["Expected_Cost_Impact"] = (
    open_trends["Cost_Impact"]
    * open_trends["Probability"]
)

print(
    "Expected schedule exposure:",
    round(open_trends["Expected_Schedule_Impact"].sum(), 2),
    "days"
)

print(
    "Expected cost exposure: S/",
    round(open_trends["Expected_Cost_Impact"].sum(), 2)
)

display(
    open_trends[
        [
            "Trend_ID",
            "Activity_ID",
            "Trend_Name",
            "Type",
            "Schedule_Impact_Days",
            "Probability",
            "Expected_Schedule_Impact",
            "Expected_Cost_Impact"
        ]
    ]
)

## 21 — Current Forecast vs Potential Forecast

In [ ]:
# ============================================================
# 21 — Current Forecast vs Potential Forecast
# ============================================================

# Último corte disponible
latest_cut = "C04"

current_row = project_controls_scorecard[
    project_controls_scorecard["Cut_ID"] == latest_cut
].iloc[0]

current_forecast = pd.to_datetime(
    current_row["Forecast_Finish"]
).normalize()

current_eac = current_row["EAC_Internal"]

# Exposición esperada de tendencias abiertas
expected_schedule_exposure = (
    open_trends["Expected_Schedule_Impact"].sum()
)

expected_cost_exposure = (
    open_trends["Expected_Cost_Impact"].sum()
)

# Fecha potencial indicativa
# Se utiliza calendario simple únicamente como indicador de exposición.
potential_forecast = (
    current_forecast
    + pd.to_timedelta(expected_schedule_exposure, unit="D")
)

potential_eac = (
    current_eac
    + expected_cost_exposure
)

forecast_summary = pd.DataFrame([
    {
        "Scenario": "BASELINE",
        "Finish_Date": pd.Timestamp(PROJECT["baseline_finish"]),
        "Finish_Variance_Days": 0,
        "Forecast_Cost": project_controls_scorecard.loc[
            project_controls_scorecard["Cut_ID"] == latest_cut,
            "BAC_Internal"
        ].iloc[0],
        "Description": "Plan aprobado / referencia"
    },
    {
        "Scenario": "CURRENT FORECAST",
        "Finish_Date": current_forecast,
        "Finish_Variance_Days": int(
            current_row["Finish_Variance_Days"]
        ),
        "Forecast_Cost": current_eac,
        "Description": "Forecast incorporado en P6 al C04"
    },
    {
        "Scenario": "POTENTIAL FORECAST",
        "Finish_Date": potential_forecast,
        "Finish_Variance_Days": (
            potential_forecast
            - pd.Timestamp(PROJECT["baseline_finish"])
        ).days,
        "Forecast_Cost": potential_eac,
        "Description": "Current Forecast + exposición esperada de Open Trends"
    }
])

display(forecast_summary.round(2))

In [ ]:
add_audit(
    audit_log,
    "POTENTIAL_FORECAST_METHOD",
    "WARNING",
    (
        "Potential Forecast es un indicador de exposición gerencial "
        "calculado como Current Forecast + impacto esperado de Open Trends. "
        "No corresponde a una reprogramación CPM ejecutada en Primavera P6."
    ),
    "FORECAST"
)

df_audit = pd.DataFrame(audit_log)

print("Método Potential Forecast registrado.")

## 22 — Recovery Readiness & Candidate Activities

In [ ]:
# ============================================================
# 22 — Recovery Readiness
# Identificación de actividades candidatas antes de optimizar
# ============================================================

latest = driver_detail[
    driver_detail["Cut_ID"] == "C04"
].copy()

# Candidatas:
# - actividades en progreso críticas o con atraso;
# - actividades futuras críticas ya desplazadas.
recovery_candidates = latest[
    (
        (latest["Execution_Status"] == "IN PROGRESS")
        & (
            latest["Critical_Flag"]
            | (latest["Activity_Finish_Variance_Days"] > 0)
        )
    )
    |
    (
        (latest["Execution_Status"] == "NOT STARTED")
        & latest["Critical_Flag"]
        & (latest["Activity_Finish_Variance_Days"] > 0)
    )
].copy()

recovery_candidates["Recovery_Class"] = np.select(
    [
        recovery_candidates["Execution_Status"].eq("IN PROGRESS"),
        recovery_candidates["Execution_Status"].eq("NOT STARTED")
    ],
    [
        "ACTIVE RECOVERY",
        "DOWNSTREAM RECOVERY"
    ],
    default="OTHER"
)

recovery_candidates = recovery_candidates.sort_values(
    [
        "Recovery_Class",
        "Activity_Finish_Variance_Days"
    ],
    ascending=[True, False]
)

display(
    recovery_candidates[
        [
            "Activity_ID",
            "Activity_Name",
            "Execution_Status",
            "Recovery_Class",
            "Physical_Pct",
            "Planned_Pct",
            "Schedule_Gap_Pct",
            "BL_Finish",
            "Forecast_Finish",
            "Activity_Finish_Variance_Days",
            "Critical_Flag",
            "Budget_Cost_Internal",
            "EAC_Cost_Internal"
        ]
    ].round(3)
)

#23 — Check de datos necesarios para NSGA-II

In [ ]:
# ============================================================
# 23 — Recovery Data Requirements
# ============================================================

recovery_readiness = pd.DataFrame([
    {
        "Input": "Current schedule C04",
        "Available": True,
        "Purpose": "Estado actual y forecast"
    },
    {
        "Input": "Activity physical progress",
        "Available": True,
        "Purpose": "Estado de ejecución"
    },
    {
        "Input": "Resource actual / remaining units",
        "Available": True,
        "Purpose": "Productividad y costo"
    },
    {
        "Input": "S10 unit prices",
        "Available": True,
        "Purpose": "Costo incremental"
    },
    {
        "Input": "Critical / total float",
        "Available": True,
        "Purpose": "Priorización"
    },
    {
        "Input": "P6 activity relationships",
        "Available": False,
        "Purpose": "Recalcular CPM y propagación del recovery"
    },
])

display(recovery_readiness)

In [ ]:
add_audit(
    audit_log,
    "RECOVERY_NETWORK_RELATIONSHIPS",
    "WARNING",
    (
        "El modelo dispone de actividades, fechas, float, recursos y costos, "
        "pero todavía no ha incorporado las relaciones lógicas de P6. "
        "No ejecutar optimización de recuperación CPM hasta integrar TASKPRED/XER."
    ),
    "RECOVERY"
)

df_audit = pd.DataFrame(audit_log)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_Controls_Recovery_Demo"
)

RAW_DIR = PROJECT_ROOT / "01_raw"
CURATED_DIR = PROJECT_ROOT / "02_curated"
OUTPUT_DIR_DRIVE = PROJECT_ROOT / "03_outputs"
AUDIT_DIR = PROJECT_ROOT / "04_audit"

for folder in [
    RAW_DIR,
    CURATED_DIR,
    OUTPUT_DIR_DRIVE,
    AUDIT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

In [ ]:
import shutil

for archivo in DATA_DIR.glob("*"):
    destino = RAW_DIR / archivo.name
    shutil.copy2(archivo, destino)
    print("✅", archivo.name)

In [ ]:
for archivo in OUTPUT_DIR.glob("*"):
    destino = CURATED_DIR / archivo.name
    shutil.copy2(archivo, destino)
    print("✅", archivo.name)

In [ ]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR existe:", RAW_DIR.exists())
print("Archivos raw:", len(list(RAW_DIR.glob("*"))))
print("Fact Activity en memoria:", "fact_activity_snapshot" in globals())
print("Fact Resource en memoria:", "fact_resource_snapshot" in globals())
print("Driver Detail en memoria:", "driver_detail" in globals())
print("Trend Register en memoria:", "trend_register" in globals())

In [ ]:
# Verificar si ya existe algún XER en la carpeta RAW

xer_files = list(RAW_DIR.glob("*.xer"))

print("XER encontrados:", len(xer_files))

for f in xer_files:
    print(" -", f.name)

In [ ]:
from google.colab import files

uploaded_xer = files.upload()

for name, content in uploaded_xer.items():
    if name.lower().endswith(".xer"):
        destino = RAW_DIR / name
        destino.write_bytes(content)
        print("✅ XER guardado en Drive:", destino)

In [ ]:
xer_files = list(RAW_DIR.glob("*.xer"))

for f in xer_files:
    print("✅", f.name, "|", f.stat().st_size, "bytes")

## 22 — P6 XER Network Integration

In [ ]:
# ============================================================
# 22.1 — Registrar e inspeccionar XER C04
# ============================================================

XER_PATH = RAW_DIR / "Linear_Infrastructure_Demo_C04_20260831.xer"

print("Existe:", XER_PATH.exists())
print("Tamaño:", XER_PATH.stat().st_size, "bytes")
print("SHA-256:", sha256_file(XER_PATH))

In [ ]:
# ============================================================
# 22.2 — Parser seguro de tablas XER
# ============================================================

import pandas as pd

def decode_xer(path):
    raw = path.read_bytes()

    # Primavera puede exportar XER con distintas codificaciones
    for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue

    return raw.decode("utf-8", errors="replace")


def parse_xer(path):
    text = decode_xer(path)
    lines = text.splitlines()

    tables = {}

    current_table = None
    current_fields = None
    current_rows = []

    def save_current():
        nonlocal current_table, current_fields, current_rows

        if current_table is not None and current_fields is not None:
            tables[current_table] = pd.DataFrame(
                current_rows,
                columns=current_fields
            )

    for line in lines:

        parts = line.split("\t")
        marker = parts[0].strip()

        if marker == "%T":
            save_current()

            current_table = parts[1].strip()
            current_fields = None
            current_rows = []

        elif marker == "%F":
            current_fields = [
                x.strip() for x in parts[1:]
            ]

        elif marker == "%R" and current_fields is not None:

            row = parts[1:]

            # Ajustar longitud si algún registro viene incompleto
            if len(row) < len(current_fields):
                row += [""] * (
                    len(current_fields) - len(row)
                )

            elif len(row) > len(current_fields):
                row = row[:len(current_fields)]

            current_rows.append(row)

        elif marker == "%E":
            save_current()
            break

    return tables


xer_tables = parse_xer(XER_PATH)

print("Número de tablas XER:", len(xer_tables))
print()

for name, df in xer_tables.items():
    print(f"{name:20s} | filas: {len(df):4d} | columnas: {len(df.columns):3d}")

In [ ]:
# ============================================================
# 22.3 — Inspección de TASK y TASKPRED
# ============================================================

print("=== TASK ===")
print(xer_tables["TASK"].columns.tolist())

print()
print("Filas TASK:", len(xer_tables["TASK"]))

display(
    xer_tables["TASK"].head()
)

print()
print("=== TASKPRED ===")
print(xer_tables["TASKPRED"].columns.tolist())

print()
print("Filas TASKPRED:", len(xer_tables["TASKPRED"]))

display(
    xer_tables["TASKPRED"].head(10)
)

In [ ]:
# ============================================================
# 22.4 — Red lógica P6 legible
# TASKPRED: predecessor -> successor
# ============================================================

task_xer = xer_tables["TASK"].copy()
pred_xer = xer_tables["TASKPRED"].copy()

# Normalizar IDs internos
for c in ["task_id"]:
    task_xer[c] = task_xer[c].astype(str).str.strip()

for c in ["task_id", "pred_task_id"]:
    pred_xer[c] = pred_xer[c].astype(str).str.strip()

# Lookup ID interno P6 -> Activity ID / nombre
task_lookup = (
    task_xer[
        [
            "task_id",
            "task_code",
            "task_name",
            "status_code",
            "clndr_id",
            "total_float_hr_cnt",
            "remain_drtn_hr_cnt",
            "target_drtn_hr_cnt"
        ]
    ]
    .copy()
)

task_lookup["task_code"] = task_lookup["task_code"].astype(str).str.strip()

# Predecesor
network_edges = pred_xer.merge(
    task_lookup[
        ["task_id", "task_code", "task_name"]
    ].rename(columns={
        "task_id": "pred_task_id",
        "task_code": "Predecessor_ID",
        "task_name": "Predecessor_Name"
    }),
    how="left",
    on="pred_task_id",
    validate="many_to_one"
)

# Sucesor
network_edges = network_edges.merge(
    task_lookup[
        ["task_id", "task_code", "task_name"]
    ].rename(columns={
        "task_id": "task_id",
        "task_code": "Successor_ID",
        "task_name": "Successor_Name"
    }),
    how="left",
    on="task_id",
    validate="many_to_one"
)

# Tipo de relación legible
REL_MAP = {
    "PR_FS": "FS",
    "PR_SS": "SS",
    "PR_FF": "FF",
    "PR_SF": "SF"
}

network_edges["Relationship_Type"] = (
    network_edges["pred_type"]
    .map(REL_MAP)
    .fillna(network_edges["pred_type"])
)

network_edges["Lag_Hours"] = pd.to_numeric(
    network_edges["lag_hr_cnt"],
    errors="coerce"
).fillna(0)

# Vista final
network_edges = network_edges[
    [
        "Predecessor_ID",
        "Predecessor_Name",
        "Successor_ID",
        "Successor_Name",
        "Relationship_Type",
        "Lag_Hours"
    ]
].copy()

print("Relaciones CPM reconstruidas:", len(network_edges))
print(
    "Relaciones sin Activity ID:",
    network_edges[
        ["Predecessor_ID", "Successor_ID"]
    ].isna().any(axis=1).sum()
)

display(network_edges.head(20))

In [ ]:
# ============================================================
# 22.5 — Relaciones directas de RC01
# ============================================================

rc01_out = network_edges[
    network_edges["Predecessor_ID"] == "RC01"
].copy()

rc01_in = network_edges[
    network_edges["Successor_ID"] == "RC01"
].copy()

print("=== PREDECESORES DIRECTOS DE RC01 ===")
display(rc01_in)

print()
print("=== SUCESORES DIRECTOS DE RC01 ===")
display(rc01_out)

In [ ]:
# ============================================================
# 22.6 — Trazabilidad de red RC01 -> MILESTONE_FINISH
# ============================================================

import networkx as nx

G = nx.DiGraph()

for _, row in network_edges.iterrows():
    G.add_edge(
        row["Predecessor_ID"],
        row["Successor_ID"],
        relationship=row["Relationship_Type"],
        lag_hours=row["Lag_Hours"]
    )

print("Nodos:", G.number_of_nodes())
print("Relaciones:", G.number_of_edges())

print(
    "¿Existe camino RC01 -> MILESTONE_FINISH?:",
    nx.has_path(G, "RC01", "MILESTONE_FINISH")
)

if nx.has_path(G, "RC01", "MILESTONE_FINISH"):

    paths = list(
        nx.all_simple_paths(
            G,
            source="RC01",
            target="MILESTONE_FINISH"
        )
    )

    print("Número de caminos RC01 -> MILESTONE_FINISH:", len(paths))
    print()

    for i, path in enumerate(paths, 1):
        print(f"PATH {i}:")
        print(" -> ".join(path))
        print()

In [ ]:
# ============================================================
# 22.7 — Critical Path Propagation Analysis
# ============================================================

# Preparar atributos numéricos de TASK
task_network = task_xer[
    [
        "task_code",
        "task_name",
        "status_code",
        "total_float_hr_cnt",
        "remain_drtn_hr_cnt",
        "target_drtn_hr_cnt",
        "driving_path_flag",
        "float_path",
        "float_path_order"
    ]
].copy()

task_network["task_code"] = (
    task_network["task_code"]
    .astype(str)
    .str.strip()
)

for c in [
    "total_float_hr_cnt",
    "remain_drtn_hr_cnt",
    "target_drtn_hr_cnt",
    "float_path",
    "float_path_order"
]:
    task_network[c] = pd.to_numeric(
        task_network[c],
        errors="coerce"
    )

task_attr = (
    task_network
    .set_index("task_code")
    .to_dict("index")
)

# Reconstruir los caminos
paths = list(
    nx.all_simple_paths(
        G,
        source="RC01",
        target="MILESTONE_FINISH"
    )
)

path_rows = []

for i, path in enumerate(paths, 1):

    node_floats = []
    driving_flags = []
    edge_lags = []
    edge_types = []

    for node in path:
        attrs = task_attr.get(node, {})

        tf = attrs.get("total_float_hr_cnt", np.nan)
        node_floats.append(tf)

        driving_flags.append(
            attrs.get("driving_path_flag", None)
        )

    for u, v in zip(path[:-1], path[1:]):
        edge = G[u][v]

        edge_lags.append(
            float(edge.get("lag_hours", 0))
        )

        edge_types.append(
            edge.get("relationship", "")
        )

    valid_floats = [
        f for f in node_floats
        if pd.notna(f)
    ]

    max_float = (
        max(valid_floats)
        if valid_floats
        else np.nan
    )

    zero_float_nodes = sum(
        abs(f) < 0.01
        for f in valid_floats
    )

    path_rows.append({
        "Path_ID": f"P{i:02d}",
        "Path": " → ".join(path),
        "Activities": len(path),
        "Max_Total_Float": max_float,
        "Zero_Float_Nodes": zero_float_nodes,
        "Total_Lag_Hours": sum(edge_lags),
        "Relationships": " → ".join(edge_types),
        "All_Nodes_Zero_Float": (
            len(valid_floats) > 0
            and all(abs(f) < 0.01 for f in valid_floats)
        )
    })

path_summary = pd.DataFrame(path_rows)

display(
    path_summary.sort_values(
        [
            "All_Nodes_Zero_Float",
            "Max_Total_Float"
        ],
        ascending=[False, True]
    )
)

In [ ]:
# Mostrar actividades involucradas en cualquier camino RC01 -> MILESTONE_FINISH

nodes_rc01_milestone_finish = sorted(
    set(
        node
        for path in paths
        for node in path
    )
)

path_node_detail = (
    task_network[
        task_network["task_code"].isin(
            nodes_rc01_milestone_finish
        )
    ]
    .rename(columns={
        "task_code": "Activity_ID",
        "task_name": "Activity_Name"
    })
    .sort_values(
        [
            "float_path",
            "float_path_order"
        ]
    )
)

display(path_node_detail)

In [ ]:
# ============================================================
# 22.9 — Inspección de calendarios P6
# ============================================================

calendar_xer = xer_tables["CALENDAR"].copy()

print("Columnas CALENDAR:")
print(calendar_xer.columns.tolist())

print()
print("Número de calendarios:", len(calendar_xer))

display(calendar_xer)

In [ ]:
# ============================================================
# 22.10 — Common Critical Chain Nodes
# ============================================================

critical_paths = [
    path
    for path, row in zip(paths, path_rows)
    if row["All_Nodes_Zero_Float"]
]

common_nodes = set(critical_paths[0])

for path in critical_paths[1:]:
    common_nodes &= set(path)

# Frecuencia de aparición en caminos críticos
node_frequency = {}

for path in critical_paths:
    for node in path:
        node_frequency[node] = (
            node_frequency.get(node, 0) + 1
        )

critical_node_summary = pd.DataFrame([
    {
        "Activity_ID": node,
        "Critical_Path_Count": count,
        "Critical_Path_Pct": (
            count / len(critical_paths)
        )
    }
    for node, count in node_frequency.items()
]).sort_values(
    ["Critical_Path_Count", "Activity_ID"],
    ascending=[False, True]
)

print("Caminos completamente críticos:", len(critical_paths))
print("Nodos comunes a TODOS:", sorted(common_nodes))

display(critical_node_summary)

In [ ]:
# ============================================================
# 22.11 — Calendar Assignment Audit
# ============================================================

task_calendar = task_xer[
    [
        "task_code",
        "task_name",
        "clndr_id",
        "status_code"
    ]
].copy()

calendar_info = calendar_xer[
    [
        "clndr_id",
        "clndr_name",
        "day_hr_cnt",
        "week_hr_cnt",
        "clndr_type",
        "base_clndr_id"
    ]
].copy()

task_calendar = task_calendar.merge(
    calendar_info,
    how="left",
    on="clndr_id",
    validate="many_to_one"
)

print("Distribución de calendarios en TASK:")

display(
    task_calendar.groupby(
        [
            "clndr_id",
            "clndr_name",
            "day_hr_cnt",
            "week_hr_cnt"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Activities")
)

print()
print("Calendarios de actividades RC01 -> MILESTONE_FINISH:")

display(
    task_calendar[
        task_calendar["task_code"].isin(
            nodes_rc01_milestone_finish
        )
    ].sort_values("task_code")
)

In [ ]:
# ============================================================
# 22.12 — Detalle del calendario crítico 6595
# ============================================================

cal_6595 = calendar_xer[
    calendar_xer["clndr_id"].astype(str).eq("6595")
].iloc[0]

calendar_raw = str(cal_6595["clndr_data"])

# P6 utiliza caracteres separadores internos.
# Los hacemos legibles sin modificar la fuente.
calendar_pretty = calendar_raw.replace("\x7f", "\n")

print("Calendario:", cal_6595["clndr_name"])
print("Horas/día:", cal_6595["day_hr_cnt"])
print("Horas/semana:", cal_6595["week_hr_cnt"])
print()
print("=== ESTRUCTURA CALENDAR DATA ===")

print("\n".join(calendar_pretty.splitlines()[:150]))

In [ ]:
# ============================================================
# 22.12.1 — Decodificar excepciones del calendario P6
# ============================================================

import re
import pandas as pd

# Extraer números d|xxxxx
exception_serials = [
    int(x)
    for x in re.findall(r"d\|(\d+)", calendar_raw)
]

print("Excepciones encontradas:", len(exception_serials))

# Conversión tipo serial Excel/P6
exception_dates = pd.to_datetime(
    exception_serials,
    unit="D",
    origin="1899-12-30"
)

calendar_exceptions = pd.DataFrame({
    "Serial": exception_serials,
    "Exception_Date": exception_dates
})

display(calendar_exceptions.tail(20))

# Excepciones relevantes para nuestro proyecto
project_exceptions = calendar_exceptions[
    calendar_exceptions["Exception_Date"].between(
        "2025-05-15",
        "2026-01-31"
    )
].copy()

print()
print(
    "Excepciones dentro del horizonte del proyecto:",
    len(project_exceptions)
)

display(project_exceptions)

In [ ]:
# ============================================================
# 22.13 — P6 Scheduling Options
# ============================================================

sched_options = xer_tables["SCHEDOPTIONS"].copy()

print("Columnas SCHEDOPTIONS:")
print(sched_options.columns.tolist())

print()
display(sched_options.T)

In [ ]:
# ============================================================
# 22.14 — Project / Data Date audit
# ============================================================

project_xer = xer_tables["PROJECT"].copy()

interesting_project_cols = [
    c for c in project_xer.columns
    if (
        "date" in c.lower()
        or "recalc" in c.lower()
        or "data" in c.lower()
    )
]

print("Campos temporales PROJECT:")

display(
    project_xer[
        interesting_project_cols
    ].T
)

## 23 — CPM Engine

In [ ]:
# ============================================================
# 23.1 — Calendario laboral P6
# L-S: 08:00-12:00 / 13:00-17:00
# Domingo + excepciones: no laborable
# ============================================================

HOLIDAYS = set(
    project_exceptions["Exception_Date"]
    .dt.normalize()
    .tolist()
)

def is_working_day(dt):
    dt = pd.Timestamp(dt)

    # Python: lunes=0 ... domingo=6
    monday_to_saturday = dt.weekday() <= 5
    holiday = dt.normalize() in HOLIDAYS

    return monday_to_saturday and not holiday


def next_working_day(dt):
    dt = pd.Timestamp(dt).normalize() + pd.Timedelta(days=1)

    while not is_working_day(dt):
        dt += pd.Timedelta(days=1)

    return dt


def align_to_work_time(dt):
    dt = pd.Timestamp(dt)

    # Día no laborable
    if not is_working_day(dt):
        d = dt.normalize()

        while not is_working_day(d):
            d += pd.Timedelta(days=1)

        return d + pd.Timedelta(hours=8)

    t = dt.time()

    # Antes de inicio
    if t < pd.Timestamp("08:00").time():
        return dt.normalize() + pd.Timedelta(hours=8)

    # Primera jornada
    if pd.Timestamp("08:00").time() <= t < pd.Timestamp("12:00").time():
        return dt

    # Refrigerio
    if pd.Timestamp("12:00").time() <= t < pd.Timestamp("13:00").time():
        return dt.normalize() + pd.Timedelta(hours=13)

    # Segunda jornada
    if pd.Timestamp("13:00").time() <= t < pd.Timestamp("17:00").time():
        return dt

    # Después de jornada
    d = next_working_day(dt)
    return d + pd.Timedelta(hours=8)


def add_work_hours(start, hours):
    dt = align_to_work_time(start)
    remaining = float(hours)

    if remaining <= 0:
        return dt

    while remaining > 1e-9:

        dt = align_to_work_time(dt)

        # Mañana
        if dt.hour < 12:
            period_end = dt.normalize() + pd.Timedelta(hours=12)

        # Tarde
        else:
            period_end = dt.normalize() + pd.Timedelta(hours=17)

        available = (
            period_end - dt
        ).total_seconds() / 3600

        if remaining <= available + 1e-9:
            return dt + pd.Timedelta(hours=remaining)

        remaining -= available

        # Al llegar a las 12:00 → 13:00
        if period_end.hour == 12:
            dt = dt.normalize() + pd.Timedelta(hours=13)

        # Al llegar a las 17:00 → próximo día hábil
        else:
            dt = next_working_day(dt) + pd.Timedelta(hours=8)

    return dt

In [ ]:
# ============================================================
# 23.2 — Unit tests básicos del calendario
# ============================================================

calendar_tests = pd.DataFrame([
    {
        "Test": "8 h desde lunes",
        "Start": pd.Timestamp("2025-09-07 08:00"),
        "Hours": 8
    },
    {
        "Test": "16 h desde lunes",
        "Start": pd.Timestamp("2025-09-07 08:00"),
        "Hours": 16
    },
    {
        "Test": "Cruzar domingo",
        "Start": pd.Timestamp("2025-09-12 08:00"),  # sábado
        "Hours": 16
    },
    {
        "Test": "Cruzar feriado 08-Oct",
        "Start": pd.Timestamp("2025-10-07 08:00"),
        "Hours": 16
    },
])

calendar_tests["Finish"] = calendar_tests.apply(
    lambda r: add_work_hours(
        r["Start"],
        r["Hours"]
    ),
    axis=1
)

display(calendar_tests)

In [ ]:
# ============================================================
# 23.3 — Estado CPM actual en P6
# ============================================================

date_cols_xer = [
    "act_start_date",
    "act_end_date",
    "early_start_date",
    "early_end_date",
    "restart_date",
    "reend_date",
    "expect_end_date",
    "target_start_date",
    "target_end_date"
]

for c in date_cols_xer:
    task_xer[c] = pd.to_datetime(
        task_xer[c],
        errors="coerce"
    )

task_xer["remain_drtn_hr_cnt"] = pd.to_numeric(
    task_xer["remain_drtn_hr_cnt"],
    errors="coerce"
)

inspect_ids = [
    "RC01",
    "RC02",
    "RC03",
    "ACT06",
    "A1190",
    "A1210",
    "A1220",
    "RC04",
    "MILESTONE_FINISH"
]

display(
    task_xer[
        task_xer["task_code"].isin(inspect_ids)
    ][
        [
            "task_code",
            "status_code",
            "act_start_date",
            "act_end_date",
            "remain_drtn_hr_cnt",
            "early_start_date",
            "early_end_date",
            "restart_date",
            "reend_date",
            "expect_end_date",
            "total_float_hr_cnt"
        ]
    ].sort_values("task_code")
)

In [ ]:
# ============================================================
# 23.4 — Diagnóstico de relaciones y restricciones
# ============================================================

print("Tipos de relación:")
display(
    network_edges["Relationship_Type"]
    .value_counts()
    .rename_axis("Relationship")
    .reset_index(name="Count")
)

print("\nRelaciones con lag distinto de cero:")
display(
    network_edges[
        network_edges["Lag_Hours"].abs() > 0.001
    ]
)

print("\nExpected Finish cargados:")
display(
    task_xer[
        task_xer["expect_end_date"].notna()
    ][["task_code", "task_name", "expect_end_date"]]
)

print("\nTipos de restricciones:")
display(
    task_xer["cstr_type"]
    .fillna("NONE")
    .value_counts()
    .rename_axis("Constraint_Type")
    .reset_index(name="Count")
)

In [ ]:
# ============================================================
# 23.5 — Restar horas laborables
# ============================================================

def previous_working_day(dt):
    dt = pd.Timestamp(dt).normalize() - pd.Timedelta(days=1)

    while not is_working_day(dt):
        dt -= pd.Timedelta(days=1)

    return dt


def align_backward_to_work_time(dt):
    dt = pd.Timestamp(dt)

    if not is_working_day(dt):
        d = dt.normalize()

        while not is_working_day(d):
            d -= pd.Timedelta(days=1)

        return d + pd.Timedelta(hours=17)

    t = dt.time()

    if t > pd.Timestamp("17:00").time():
        return dt.normalize() + pd.Timedelta(hours=17)

    if pd.Timestamp("13:00").time() < t <= pd.Timestamp("17:00").time():
        return dt

    if pd.Timestamp("12:00").time() < t <= pd.Timestamp("13:00").time():
        return dt.normalize() + pd.Timedelta(hours=12)

    if pd.Timestamp("08:00").time() < t <= pd.Timestamp("12:00").time():
        return dt

    # 08:00 o antes → fin del día hábil anterior
    d = previous_working_day(dt)
    return d + pd.Timedelta(hours=17)


def subtract_work_hours(finish, hours):
    dt = pd.Timestamp(finish)
    remaining = float(hours)

    if remaining <= 0:
        return dt

    while remaining > 1e-9:

        dt = align_backward_to_work_time(dt)

        # Tramo de tarde
        if dt.hour > 13 or (
            dt.hour == 13 and dt.minute > 0
        ):
            period_start = dt.normalize() + pd.Timedelta(hours=13)

        # Tramo de mañana
        else:
            period_start = dt.normalize() + pd.Timedelta(hours=8)

        available = (
            dt - period_start
        ).total_seconds() / 3600

        if remaining <= available + 1e-9:
            return dt - pd.Timedelta(hours=remaining)

        remaining -= available

        # Si llegamos a 13:00, saltamos a 12:00
        if period_start.hour == 13:
            dt = dt.normalize() + pd.Timedelta(hours=12)

        else:
            d = previous_working_day(dt)
            dt = d + pd.Timedelta(hours=17)

    return dt


def shift_work_hours(dt, hours):
    if hours >= 0:
        return add_work_hours(dt, hours)
    else:
        return subtract_work_hours(dt, abs(hours))

In [ ]:
# ============================================================
# 23.6 — Forward Pass CPM
# C04 / Retained Logic simplificado
# ============================================================

DATA_DATE = (
    pd.Timestamp(CUTS["C04"]).normalize()
    + pd.Timedelta(hours=8)
)

print("Data Date CPM:", DATA_DATE)

# ------------------------------------------------------------
# Preparar información de actividades
# ------------------------------------------------------------

cpm_tasks = task_xer[
    [
        "task_code",
        "task_name",
        "status_code",
        "task_type",
        "remain_drtn_hr_cnt",
        "act_start_date",
        "act_end_date",
        "restart_date",
        "reend_date",
        "early_start_date",
        "early_end_date"
    ]
].copy()

cpm_tasks["task_code"] = (
    cpm_tasks["task_code"]
    .astype(str)
    .str.strip()
)

cpm_tasks = cpm_tasks.set_index("task_code")

# ------------------------------------------------------------
# Forward Pass
# ------------------------------------------------------------

results = {}

topological_order = list(nx.topological_sort(G))

for activity in topological_order:

    row = cpm_tasks.loc[activity]

    status = row["status_code"]

    duration = pd.to_numeric(
        row["remain_drtn_hr_cnt"],
        errors="coerce"
    )

    duration = (
        0.0
        if pd.isna(duration)
        else float(duration)
    )

    # ========================================================
    # ACTIVIDAD COMPLETADA
    # ========================================================

    if status == "TK_Complete":

        es = pd.Timestamp(row["act_start_date"])
        ef = pd.Timestamp(row["act_end_date"])

        results[activity] = {
            "CPM_Start": es,
            "CPM_Finish": ef,
            "Status": status
        }

        continue

    # ========================================================
    # BASE EARLY START
    # ========================================================

    if status == "TK_Active":

        # Segmento remanente
        if pd.notna(row["restart_date"]):
            base_start = pd.Timestamp(
                row["restart_date"]
            )
        else:
            base_start = DATA_DATE

    else:

        # Actividad todavía no iniciada
        base_start = DATA_DATE

    candidate_starts = [base_start]

    # ========================================================
    # RELACIONES DE PRECEDENCIA
    # ========================================================

    incoming = network_edges[
        network_edges["Successor_ID"] == activity
    ]

    for _, rel in incoming.iterrows():

        pred = rel["Predecessor_ID"]

        # Si por alguna razón el predecesor no fue calculado
        if pred not in results:
            continue

        pred_start = results[pred]["CPM_Start"]
        pred_finish = results[pred]["CPM_Finish"]

        rel_type = rel["Relationship_Type"]
        lag = float(rel["Lag_Hours"])

        # ------------------------
        # Finish-to-Start
        # ------------------------
        if rel_type == "FS":

            constraint_start = shift_work_hours(
                pred_finish,
                lag
            )

            # FS con lag 0 desde las 17:00 debe pasar
            # al próximo instante laboral.
            if lag == 0:
                constraint_start = align_to_work_time(
                    pred_finish
                )

        # ------------------------
        # Start-to-Start
        # ------------------------
        elif rel_type == "SS":

            constraint_start = shift_work_hours(
                pred_start,
                lag
            )

        # ------------------------
        # Finish-to-Finish
        # ------------------------
        elif rel_type == "FF":

            required_finish = shift_work_hours(
                pred_finish,
                lag
            )

            constraint_start = subtract_work_hours(
                required_finish,
                duration
            )

        # ------------------------
        # Start-to-Finish
        # ------------------------
        elif rel_type == "SF":

            required_finish = shift_work_hours(
                pred_start,
                lag
            )

            constraint_start = subtract_work_hours(
                required_finish,
                duration
            )

        else:
            continue

        candidate_starts.append(
            constraint_start
        )

    # Mayor restricción de inicio
    es = max(candidate_starts)

    # Alinear solamente actividades con duración > 0
    if duration > 0:
        es = align_to_work_time(es)
        ef = add_work_hours(es, duration)

    else:
        # Hitos: conservar exactamente el instante lógico
        ef = es

    results[activity] = {
        "CPM_Start": es,
        "CPM_Finish": ef,
        "Status": status
    }


cpm_result = (
    pd.DataFrame.from_dict(
        results,
        orient="index"
    )
    .reset_index()
    .rename(columns={"index": "Activity_ID"})
)

display(cpm_result.head())

In [ ]:
# ============================================================
# 23.7 — Validación CPM Python vs Primavera P6
# ============================================================

p6_benchmark = task_xer[
    [
        "task_code",
        "status_code",
        "restart_date",
        "reend_date"
    ]
].copy()

p6_benchmark = p6_benchmark.rename(columns={
    "task_code": "Activity_ID",
    "restart_date": "P6_Start",
    "reend_date": "P6_Finish"
})

cpm_validation = cpm_result.merge(
    p6_benchmark,
    how="left",
    on="Activity_ID",
    validate="one_to_one"
)

cpm_validation["Start_Delta_Hours"] = (
    cpm_validation["CPM_Start"]
    - cpm_validation["P6_Start"]
).dt.total_seconds() / 3600

cpm_validation["Finish_Delta_Hours"] = (
    cpm_validation["CPM_Finish"]
    - cpm_validation["P6_Finish"]
).dt.total_seconds() / 3600

inspect_validation = cpm_validation[
    cpm_validation["Activity_ID"].isin(
        [
            "RC01",
            "RC02",
            "RC03",
            "ACT06",
            "A1190",
            "A1210",
            "A1220",
            "RC04",
            "MILESTONE_FINISH"
        ]
    )
]

display(
    inspect_validation[
        [
            "Activity_ID",
            "CPM_Start",
            "P6_Start",
            "Start_Delta_Hours",
            "CPM_Finish",
            "P6_Finish",
            "Finish_Delta_Hours"
        ]
    ]
)

In [ ]:
milestone_finish_test = cpm_validation[
    cpm_validation["Activity_ID"] == "MILESTONE_FINISH"
].iloc[0]

print("=== MILESTONE_FINISH VALIDATION ===")
print("Python CPM:", milestone_finish_test["CPM_Finish"])
print("Primavera P6:", milestone_finish_test["P6_Finish"])
print(
    "Delta horas:",
    milestone_finish_test["Finish_Delta_Hours"]
)

In [ ]:
# ============================================================
# 23.8 — Localizar primer desfase Python vs P6
# ============================================================

critical_check_ids = [
    "RC01",
    "RC02",
    "RC03",
    "A1200",
    "ACT06",
    "A1190",
    "A1210",
    "A1220",
    "ACT07",
    "RC04",
    "MILESTONE_FINISH"
]

delta_check = (
    cpm_validation[
        cpm_validation["Activity_ID"].isin(critical_check_ids)
    ]
    [
        [
            "Activity_ID",
            "CPM_Start",
            "P6_Start",
            "Start_Delta_Hours",
            "CPM_Finish",
            "P6_Finish",
            "Finish_Delta_Hours"
        ]
    ]
    .copy()
)

# Orden aproximado según fechas P6
delta_check = delta_check.sort_values(
    ["P6_Start", "P6_Finish"]
)

display(delta_check)

In [ ]:
# Primera actividad donde aparece diferencia de Finish

first_finish_delta = delta_check[
    delta_check["Finish_Delta_Hours"].abs() > 0.001
]

if len(first_finish_delta):
    first_id = first_finish_delta.iloc[0]["Activity_ID"]
    print("Primer desfase detectado en:", first_id)

    print("\nRelaciones de entrada:")
    display(
        network_edges[
            network_edges["Successor_ID"] == first_id
        ]
    )

    print("\nRelaciones de salida:")
    display(
        network_edges[
            network_edges["Predecessor_ID"] == first_id
        ]
    )
else:
    print("No hay diferencias de Finish en la cadena revisada.")

In [ ]:
# ============================================================
# 23.9 — Diferencia en horas LABORABLES
# ============================================================

def working_hours_between(start, finish):
    start = pd.Timestamp(start)
    finish = pd.Timestamp(finish)

    if pd.isna(start) or pd.isna(finish):
        return np.nan

    if start == finish:
        return 0.0

    sign = 1

    if finish < start:
        start, finish = finish, start
        sign = -1

    total = 0.0
    current = start.normalize()

    while current <= finish.normalize():

        if is_working_day(current):

            periods = [
                (
                    current + pd.Timedelta(hours=8),
                    current + pd.Timedelta(hours=12)
                ),
                (
                    current + pd.Timedelta(hours=13),
                    current + pd.Timedelta(hours=17)
                )
            ]

            for p_start, p_finish in periods:

                overlap_start = max(start, p_start)
                overlap_finish = min(finish, p_finish)

                if overlap_finish > overlap_start:
                    total += (
                        overlap_finish - overlap_start
                    ).total_seconds() / 3600

        current += pd.Timedelta(days=1)

    return sign * total

In [ ]:
p6_milestone_finish = pd.Timestamp("2026-01-19 17:00")
python_milestone_finish = pd.Timestamp("2026-01-20 08:00")

print(
    "Delta calendario:",
    (python_milestone_finish - p6_milestone_finish).total_seconds() / 3600,
    "h"
)

print(
    "Delta laborable:",
    working_hours_between(p6_milestone_finish, python_milestone_finish),
    "h"
)

## 24 — Recovery Candidate Baseline

In [ ]:
# ============================================================
# 24 — Recovery Candidate Baseline
# Nodos comunes de todos los caminos críticos
# ============================================================

RECOVERY_NODES = sorted(
    set(common_nodes) - {"MILESTONE_FINISH"}
)

# Datos C04 de costo/avance
c04_evm = fact_evm[
    fact_evm["Cut_ID"] == "C04"
][
    [
        "Activity_ID",
        "Activity_Name",
        "Physical_Pct",
        "Budget_Cost_Internal",
        "Actual_Cost_Internal",
        "Remaining_Cost_Internal",
        "EAC_Cost_Internal",
        "Forecast_Finish",
        "total_float_hr_cnt",
        "remain_drtn_hr_cnt"
    ]
].copy()

# Frecuencia dentro de caminos críticos
critical_frequency = critical_node_summary[
    [
        "Activity_ID",
        "Critical_Path_Count",
        "Critical_Path_Pct"
    ]
].copy()

# Cantidad de recursos involucrados y unidades remanentes
c04_resources = (
    fact_resource_snapshot[
        fact_resource_snapshot["Cut_ID"] == "C04"
    ]
    .groupby("Activity_ID", as_index=False)
    .agg(
        Resource_Count=("Resource_ID", "nunique"),
        Remaining_Resource_Units=("remain_qty", "sum")
    )
)

recovery_baseline = (
    c04_evm[
        c04_evm["Activity_ID"].isin(RECOVERY_NODES)
    ]
    .merge(
        critical_frequency,
        how="left",
        on="Activity_ID"
    )
    .merge(
        c04_resources,
        how="left",
        on="Activity_ID"
    )
)

# Remaining Duration en jornadas de 8h
recovery_baseline["Remaining_Days_8h"] = (
    recovery_baseline["remain_drtn_hr_cnt"] / 8
)

# Costo remanente relativo
recovery_baseline["Remaining_Cost_Pct"] = (
    recovery_baseline["Remaining_Cost_Internal"]
    / recovery_baseline["Budget_Cost_Internal"]
)

# Orden natural de prioridad
recovery_baseline = recovery_baseline.sort_values(
    [
        "Critical_Path_Pct",
        "Remaining_Days_8h"
    ],
    ascending=[False, False]
)

display(
    recovery_baseline[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Remaining_Days_8h",
            "Budget_Cost_Internal",
            "Actual_Cost_Internal",
            "Remaining_Cost_Internal",
            "EAC_Cost_Internal",
            "Resource_Count",
            "Critical_Path_Count",
            "Critical_Path_Pct",
            "Forecast_Finish"
        ]
    ].round(3)
)

In [ ]:
# ============================================================
# 25 — CPM Validation Audit
# ============================================================

p6_milestone_finish = pd.Timestamp(
    cpm_validation.loc[
        cpm_validation["Activity_ID"] == "MILESTONE_FINISH",
        "P6_Finish"
    ].iloc[0]
)

python_milestone_finish = pd.Timestamp(
    cpm_validation.loc[
        cpm_validation["Activity_ID"] == "MILESTONE_FINISH",
        "CPM_Finish"
    ].iloc[0]
)

calendar_delta = (
    python_milestone_finish - p6_milestone_finish
).total_seconds() / 3600

working_delta = working_hours_between(
    p6_milestone_finish,
    python_milestone_finish
)

add_audit(
    audit_log,
    "CPM_ENGINE_VALIDATION",
    "PASS" if abs(working_delta) < 0.001 else "WARNING",
    (
        f"P6 MILESTONE_FINISH={p6_milestone_finish}; Python MILESTONE_FINISH={python_milestone_finish}; "
        f"delta calendario={calendar_delta:.1f} h; "
        f"delta laborable={working_delta:.1f} h."
    ),
    "CPM"
)

df_audit = pd.DataFrame(audit_log)

print("P6 MILESTONE_FINISH:", p6_milestone_finish)
print("Python MILESTONE_FINISH:", python_milestone_finish)
print("Delta calendario:", calendar_delta, "h")
print("Delta laborable:", working_delta, "h")

## Check Point completo

In [ ]:
# ============================================================
# 26 — Persistent Recovery Checkpoint
# ============================================================

from pathlib import Path
import hashlib

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_Controls_Recovery_Demo"
)

CURATED_DIR = PROJECT_ROOT / "02_curated"
OUTPUT_DIR_DRIVE = PROJECT_ROOT / "03_outputs"
AUDIT_DIR = PROJECT_ROOT / "04_audit"

for d in [
    CURATED_DIR,
    OUTPUT_DIR_DRIVE,
    AUDIT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)


checkpoint_tables = {

    "fact_evm.csv":
        fact_evm,

    "evm_full_summary.csv":
        evm_full_summary,

    "project_controls_scorecard.csv":
        project_controls_scorecard,

    "schedule_summary.csv":
        schedule_summary,

    "trend_register.csv":
        trend_register,

    "forecast_summary.csv":
        forecast_summary,

    "network_edges.csv":
        network_edges,

    "critical_path_summary.csv":
        path_summary,

    "critical_node_summary.csv":
        critical_node_summary,

    "recovery_baseline.csv":
        recovery_baseline,

    "cpm_validation.csv":
        cpm_validation
}

for filename, df in checkpoint_tables.items():

    path = CURATED_DIR / filename

    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    print("✅", filename)


# Audit separado
df_audit.to_csv(
    AUDIT_DIR / "audit_log.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ audit_log.csv")


# Registrar también el XER
xer_hash = sha256_file(XER_PATH)

xer_manifest = pd.DataFrame([{
    "filename": XER_PATH.name,
    "sha256": xer_hash,
    "size_bytes": XER_PATH.stat().st_size,
    "source": "Primavera P6 C04 XER",
    "data_date": "2025-08-31"
}])

xer_manifest.to_csv(
    AUDIT_DIR / "xer_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ xer_manifest.csv")
print()
print("Checkpoint guardado en Drive.")

In [ ]:
print("=== CHECKPOINT ===")

print(
    "Curated:",
    len(list(CURATED_DIR.glob("*.csv"))),
    "archivos"
)

print(
    "Audit:",
    len(list(AUDIT_DIR.glob("*"))),
    "archivos"
)

print(
    "Recovery candidates:",
    recovery_baseline["Activity_ID"].tolist()
)

In [ ]:
# ============================================================
# RESUME SESSION — Project Controls Recovery Demo
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_Controls_Recovery_Demo"
)

RAW_DIR = PROJECT_ROOT / "01_raw"
CURATED_DIR = PROJECT_ROOT / "02_curated"
OUTPUT_DIR = PROJECT_ROOT / "03_outputs"
AUDIT_DIR = PROJECT_ROOT / "04_audit"

# Tablas persistidas
fact_evm = pd.read_csv(
    CURATED_DIR / "fact_evm.csv",
    parse_dates=[
        "Snapshot_Date",
        "Forecast_Finish",
        "Actual_Start",
        "Actual_Finish"
    ]
)

evm_full_summary = pd.read_csv(
    CURATED_DIR / "evm_full_summary.csv",
    parse_dates=["Snapshot_Date"]
)

project_controls_scorecard = pd.read_csv(
    CURATED_DIR / "project_controls_scorecard.csv",
    parse_dates=[
        "Snapshot_Date",
        "Baseline_Finish",
        "Forecast_Finish"
    ]
)

schedule_summary = pd.read_csv(
    CURATED_DIR / "schedule_summary.csv",
    parse_dates=[
        "Snapshot_Date",
        "Baseline_Finish",
        "Forecast_Finish"
    ]
)

trend_register = pd.read_csv(
    CURATED_DIR / "trend_register.csv"
)

forecast_summary = pd.read_csv(
    CURATED_DIR / "forecast_summary.csv",
    parse_dates=["Finish_Date"]
)

network_edges = pd.read_csv(
    CURATED_DIR / "network_edges.csv"
)

path_summary = pd.read_csv(
    CURATED_DIR / "critical_path_summary.csv"
)

critical_node_summary = pd.read_csv(
    CURATED_DIR / "critical_node_summary.csv"
)

recovery_baseline = pd.read_csv(
    CURATED_DIR / "recovery_baseline.csv",
    parse_dates=["Forecast_Finish"]
)

cpm_validation = pd.read_csv(
    CURATED_DIR / "cpm_validation.csv",
    parse_dates=[
        "CPM_Start",
        "CPM_Finish",
        "P6_Start",
        "P6_Finish"
    ]
)

df_audit = pd.read_csv(
    AUDIT_DIR / "audit_log.csv"
)

# Fact Resource Snapshot también lo necesitamos ahora
fact_resource_snapshot = pd.read_csv(
    CURATED_DIR / "fact_resource_snapshot.csv",
    low_memory=False
)

print("✅ Sesión reconstruida")
print("Recovery candidates:",
      recovery_baseline["Activity_ID"].tolist())
print("Network relationships:", len(network_edges))
print("Audit records:", len(df_audit))

In [ ]:
# ============================================================
# 27 — Recovery Resource Profile
# ============================================================

RECOVERY_NODES = [
    "RC01",
    "RC02",
    "RC03",
    "RC04"
]

c04_recovery_resources = (
    fact_resource_snapshot[
        (fact_resource_snapshot["Cut_ID"] == "C04")
        &
        (fact_resource_snapshot["Activity_ID"].isin(RECOVERY_NODES))
    ]
    .copy()
)

# Asegurar numéricos
numeric_cols = [
    "target_qty",
    "act_qty",
    "remain_qty",
    "total_qty",
    "Precio_unit",
    "Budget_Cost_Internal",
    "Actual_Cost_Internal",
    "Remaining_Cost_Internal",
    "EAC_Cost_Internal"
]

for c in numeric_cols:
    if c in c04_recovery_resources.columns:
        c04_recovery_resources[c] = pd.to_numeric(
            c04_recovery_resources[c],
            errors="coerce"
        )

display(
    c04_recovery_resources[
        [
            "Activity_ID",
            "Resource_ID",
            "Recurso",
            "Unidad",
            "target_qty",
            "act_qty",
            "remain_qty",
            "Precio_unit",
            "Remaining_Cost_Internal"
        ]
    ]
    .sort_values(
        ["Activity_ID", "Remaining_Cost_Internal"],
        ascending=[True, False]
    )
)

In [ ]:
recovery_resource_summary = (
    c04_recovery_resources
    .groupby("Activity_ID", as_index=False)
    .agg(
        Resources=("Resource_ID", "nunique"),
        Budget_Resource_Cost=("Budget_Cost_Internal", "sum"),
        Actual_Resource_Cost=("Actual_Cost_Internal", "sum"),
        Remaining_Resource_Cost=("Remaining_Cost_Internal", "sum"),
        EAC_Resource_Cost=("EAC_Cost_Internal", "sum")
    )
)

recovery_resource_summary = recovery_resource_summary.merge(
    recovery_baseline[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Remaining_Days_8h",
            "Critical_Path_Pct"
        ]
    ],
    on="Activity_ID",
    how="left"
)

display(
    recovery_resource_summary[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Remaining_Days_8h",
            "Resources",
            "Remaining_Resource_Cost",
            "Critical_Path_Pct"
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# 27.1 — Corrección de Remaining Duration
# El Excel P6 ya entrega la duración remanente en días.
# ============================================================

recovery_baseline["Remaining_Days"] = (
    recovery_baseline["remain_drtn_hr_cnt"]
)

# Eliminar columna anterior si existe
if "Remaining_Days_8h" in recovery_baseline.columns:
    recovery_baseline = recovery_baseline.drop(
        columns=["Remaining_Days_8h"]
    )

display(
    recovery_baseline[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Remaining_Days",
            "Remaining_Cost_Internal",
            "Critical_Path_Pct"
        ]
    ].sort_values("Remaining_Days", ascending=False)
)

In [ ]:
recovery_resource_summary = (
    recovery_resource_summary
    .drop(columns=["Remaining_Days_8h"], errors="ignore")
    .merge(
        recovery_baseline[
            ["Activity_ID", "Remaining_Days"]
        ],
        on="Activity_ID",
        how="left"
    )
)

display(
    recovery_resource_summary[
        [
            "Activity_ID",
            "Activity_Name",
            "Physical_Pct",
            "Remaining_Days",
            "Resources",
            "Remaining_Resource_Cost",
            "Critical_Path_Pct"
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# 27.2 — Composición de recursos de candidatos Recovery
# ============================================================

resource_recovery_detail = (
    c04_recovery_resources[
        [
            "Activity_ID",
            "Resource_ID",
            "Recurso",
            "Unidad",
            "target_qty",
            "act_qty",
            "remain_qty",
            "Precio_unit",
            "Remaining_Cost_Internal"
        ]
    ]
    .copy()
)

resource_recovery_detail["Cost_Share"] = (
    resource_recovery_detail["Remaining_Cost_Internal"]
    /
    resource_recovery_detail.groupby("Activity_ID")[
        "Remaining_Cost_Internal"
    ].transform("sum")
)

display(
    resource_recovery_detail.sort_values(
        ["Activity_ID", "Remaining_Cost_Internal"],
        ascending=[True, False]
    ).round({
        "remain_qty": 2,
        "Precio_unit": 2,
        "Remaining_Cost_Internal": 2,
        "Cost_Share": 3
    })
)

## 28 — Recovery Cost Model


In [ ]:
# ============================================================
# 28.1 — Recursos sujetos a costo de aceleración
# ============================================================

# No todos los recursos aumentan al acelerar.
# Materiales de cantidad fija quedan fuera del crash premium.

RECOVERY_RESOURCE_UNITS = {
    "RC01": ["hh", "hm", "gal"],  # demo: labor + equipment + fuel
    "RC02": ["hh"],               # demo: labor
    "RC03": ["hh", "hm"],         # demo: labor + equipment
    "RC04": ["hh"]                # demo: labor
}

recovery_cost_rows = []

for activity, units in RECOVERY_RESOURCE_UNITS.items():

    temp = resource_recovery_detail[
        (resource_recovery_detail["Activity_ID"] == activity)
        &
        (resource_recovery_detail["Unidad"].isin(units))
    ].copy()

    recovery_cost_rows.append({
        "Activity_ID": activity,
        "Accelerable_Cost_Base":
            temp["Remaining_Cost_Internal"].sum(),
        "Accelerable_Resources":
            temp["Resource_ID"].nunique()
    })

recovery_cost_base = pd.DataFrame(
    recovery_cost_rows
)

recovery_cost_base = recovery_cost_base.merge(
    recovery_baseline[
        [
            "Activity_ID",
            "Activity_Name",
            "Remaining_Days",
            "Remaining_Cost_Internal"
        ]
    ],
    on="Activity_ID",
    how="left"
)

recovery_cost_base["Accelerable_Cost_Pct"] = (
    recovery_cost_base["Accelerable_Cost_Base"]
    /
    recovery_cost_base["Remaining_Cost_Internal"]
)

display(
    recovery_cost_base.round(3)
)

In [ ]:
# ============================================================
# 29 — Recovery Option Matrix
# ============================================================

import math

GAMMA = 0.08

RECOVERY_CONFIG = {

    "RC01": {
        "efficiency": 0.85,
        "intensities": [
            ("NORMAL", 0.00),
            ("MODERATE", 0.25),
            ("AGGRESSIVE", 0.50)
        ]
    },

    "RC02": {
        "efficiency": 0.90,
        "intensities": [
            ("NORMAL", 0.00),
            ("MODERATE", 0.50),
            ("AGGRESSIVE", 1.00)
        ]
    },

    "RC03": {
        "efficiency": 0.70,
        "intensities": [
            ("NORMAL", 0.00),
            ("MODERATE", 0.50),
            ("AGGRESSIVE", 1.00)
        ]
    },

    "RC04": {
        "efficiency": 0.90,
        "intensities": [
            ("NORMAL", 0.00),
            ("MODERATE", 0.50),
            ("AGGRESSIVE", 1.00)
        ]
    }
}

option_rows = []

for _, row in recovery_cost_base.iterrows():

    activity = row["Activity_ID"]

    D = float(
        row["Remaining_Days"]
    )

    cost_base = float(
        row["Accelerable_Cost_Base"]
    )

    eta = RECOVERY_CONFIG[
        activity
    ]["efficiency"]

    for level, intensity in RECOVERY_CONFIG[
        activity
    ]["intensities"]:

        if intensity == 0:

            new_days = int(D)
            accel_variable_cost = cost_base
            incremental_cost = 0.0

        else:

            # Capacidad efectiva:
            # 1 + intensidad × eficiencia
            effective_capacity = (
                1
                +
                intensity * eta
            )

            new_days = math.ceil(
                D / effective_capacity
            )

            # Costo diario original
            base_daily_cost = (
                cost_base / D
            )

            # Recursos adicionales incluyen
            # premium GAMMA
            accelerated_daily_cost = (
                base_daily_cost
                *
                (
                    1
                    +
                    intensity * (1 + GAMMA)
                )
            )

            accel_variable_cost = (
                accelerated_daily_cost
                * new_days
            )

            incremental_cost = max(
                0,
                accel_variable_cost
                - cost_base
            )

        days_recovered = (
            D - new_days
        )

        option_rows.append({

            "Activity_ID":
                activity,

            "Recovery_Level":
                level,

            "Extra_Capacity_Pct":
                intensity,

            "Efficiency":
                eta,

            "Original_RD_Days":
                D,

            "New_RD_Days":
                new_days,

            "New_RD_Hours":
                new_days * 8,

            "Days_Recovered":
                days_recovered,

            "Accelerable_Cost_Base":
                cost_base,

            "Incremental_Cost":
                incremental_cost,

            "Gamma":
                GAMMA
        })


recovery_options = pd.DataFrame(
    option_rows
)

display(
    recovery_options[
        [
            "Activity_ID",
            "Recovery_Level",
            "Extra_Capacity_Pct",
            "Original_RD_Days",
            "New_RD_Days",
            "Days_Recovered",
            "Incremental_Cost"
        ]
    ].round(2)
)

## 30 — Exhaustive Recovery Scenario Engine

In [ ]:
# ============================================================
# 30.1 — Recovery CPM Bootstrap
# ============================================================

from itertools import product
import re
import networkx as nx
import pandas as pd
import numpy as np

XER_PATH = RAW_DIR / "Linear_Infrastructure_Demo_C04_20260831.xer"

# ------------------------------------------------------------
# Recuperar XER si task_xer no existe en memoria
# ------------------------------------------------------------

if "task_xer" not in globals():

    def decode_xer(path):
        raw = path.read_bytes()

        for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
            try:
                return raw.decode(encoding)
            except UnicodeDecodeError:
                continue

        return raw.decode("utf-8", errors="replace")


    def parse_xer(path):
        text = decode_xer(path)
        lines = text.splitlines()

        tables = {}

        current_table = None
        current_fields = None
        current_rows = []

        def save_current():
            nonlocal current_table, current_fields, current_rows

            if current_table is not None and current_fields is not None:
                tables[current_table] = pd.DataFrame(
                    current_rows,
                    columns=current_fields
                )

        for line in lines:

            parts = line.split("\t")
            marker = parts[0].strip()

            if marker == "%T":

                save_current()

                current_table = parts[1].strip()
                current_fields = None
                current_rows = []

            elif marker == "%F":

                current_fields = [
                    x.strip() for x in parts[1:]
                ]

            elif marker == "%R" and current_fields is not None:

                row = parts[1:]

                if len(row) < len(current_fields):
                    row += [""] * (
                        len(current_fields) - len(row)
                    )

                elif len(row) > len(current_fields):
                    row = row[:len(current_fields)]

                current_rows.append(row)

            elif marker == "%E":

                save_current()
                break

        return tables


    xer_tables = parse_xer(XER_PATH)

    task_xer = xer_tables["TASK"].copy()
    calendar_xer = xer_tables["CALENDAR"].copy()

else:

    if "xer_tables" not in globals():
        xer_tables = parse_xer(XER_PATH)

    calendar_xer = xer_tables["CALENDAR"].copy()


# ------------------------------------------------------------
# Fechas / números TASK
# ------------------------------------------------------------

date_cols = [
    "act_start_date",
    "act_end_date",
    "early_start_date",
    "early_end_date",
    "restart_date",
    "reend_date",
    "expect_end_date",
    "target_start_date",
    "target_end_date"
]

for c in date_cols:
    task_xer[c] = pd.to_datetime(
        task_xer[c],
        errors="coerce"
    )

task_xer["remain_drtn_hr_cnt"] = pd.to_numeric(
    task_xer["remain_drtn_hr_cnt"],
    errors="coerce"
)

task_xer["task_code"] = (
    task_xer["task_code"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Project calendar (public reusable version)
# Use the calendar assigned to the largest number of activities.
# ------------------------------------------------------------

project_calendar_id = (
    task_xer["clndr_id"]
    .astype(str)
    .mode()
    .iloc[0]
)

project_calendar = calendar_xer[
    calendar_xer["clndr_id"]
    .astype(str)
    .eq(project_calendar_id)
].iloc[0]

calendar_raw = str(
    project_calendar["clndr_data"]
)

exception_serials = [
    int(x)
    for x in re.findall(
        r"d\|(\d+)",
        calendar_raw
    )
]

exception_dates = pd.to_datetime(
    exception_serials,
    unit="D",
    origin="1899-12-30"
)

HOLIDAYS = set(
    exception_dates.normalize().tolist()
)


# ------------------------------------------------------------
# Calendario laboral
# ------------------------------------------------------------

def is_working_day(dt):

    dt = pd.Timestamp(dt)

    return (
        dt.weekday() <= 5
        and
        dt.normalize() not in HOLIDAYS
    )


def next_working_day(dt):

    dt = (
        pd.Timestamp(dt).normalize()
        + pd.Timedelta(days=1)
    )

    while not is_working_day(dt):
        dt += pd.Timedelta(days=1)

    return dt


def previous_working_day(dt):

    dt = (
        pd.Timestamp(dt).normalize()
        - pd.Timedelta(days=1)
    )

    while not is_working_day(dt):
        dt -= pd.Timedelta(days=1)

    return dt


def align_to_work_time(dt):

    dt = pd.Timestamp(dt)

    if not is_working_day(dt):

        d = dt.normalize()

        while not is_working_day(d):
            d += pd.Timedelta(days=1)

        return d + pd.Timedelta(hours=8)

    t = dt.time()

    if t < pd.Timestamp("08:00").time():
        return dt.normalize() + pd.Timedelta(hours=8)

    if t < pd.Timestamp("12:00").time():
        return dt

    if t < pd.Timestamp("13:00").time():
        return dt.normalize() + pd.Timedelta(hours=13)

    if t < pd.Timestamp("17:00").time():
        return dt

    return (
        next_working_day(dt)
        + pd.Timedelta(hours=8)
    )


def add_work_hours(start, hours):

    dt = align_to_work_time(start)
    remaining = float(hours)

    if remaining <= 0:
        return dt

    while remaining > 1e-9:

        dt = align_to_work_time(dt)

        if dt.hour < 12:

            period_end = (
                dt.normalize()
                + pd.Timedelta(hours=12)
            )

        else:

            period_end = (
                dt.normalize()
                + pd.Timedelta(hours=17)
            )

        available = (
            period_end - dt
        ).total_seconds() / 3600

        if remaining <= available + 1e-9:

            return (
                dt
                + pd.Timedelta(hours=remaining)
            )

        remaining -= available

        if period_end.hour == 12:

            dt = (
                dt.normalize()
                + pd.Timedelta(hours=13)
            )

        else:

            dt = (
                next_working_day(dt)
                + pd.Timedelta(hours=8)
            )

    return dt


def subtract_work_hours(finish, hours):

    dt = pd.Timestamp(finish)
    remaining = float(hours)

    if remaining <= 0:
        return dt

    while remaining > 1e-9:

        if not is_working_day(dt):

            dt = (
                previous_working_day(dt)
                + pd.Timedelta(hours=17)
            )

        if dt.time() > pd.Timestamp("17:00").time():

            dt = (
                dt.normalize()
                + pd.Timedelta(hours=17)
            )

        elif (
            pd.Timestamp("12:00").time()
            < dt.time()
            <= pd.Timestamp("13:00").time()
        ):

            dt = (
                dt.normalize()
                + pd.Timedelta(hours=12)
            )

        elif dt.time() <= pd.Timestamp("08:00").time():

            dt = (
                previous_working_day(dt)
                + pd.Timedelta(hours=17)
            )

        if dt.hour > 13:

            period_start = (
                dt.normalize()
                + pd.Timedelta(hours=13)
            )

        else:

            period_start = (
                dt.normalize()
                + pd.Timedelta(hours=8)
            )

        available = (
            dt - period_start
        ).total_seconds() / 3600

        if remaining <= available + 1e-9:

            return (
                dt
                - pd.Timedelta(hours=remaining)
            )

        remaining -= available

        if period_start.hour == 13:

            dt = (
                dt.normalize()
                + pd.Timedelta(hours=12)
            )

        else:

            dt = (
                previous_working_day(dt)
                + pd.Timedelta(hours=17)
            )

    return dt


def shift_work_hours(dt, hours):

    if hours >= 0:
        return add_work_hours(dt, hours)

    return subtract_work_hours(
        dt,
        abs(hours)
    )


def working_hours_between(start, finish):

    start = pd.Timestamp(start)
    finish = pd.Timestamp(finish)

    if start == finish:
        return 0.0

    sign = 1

    if finish < start:
        start, finish = finish, start
        sign = -1

    total = 0.0
    current = start.normalize()

    while current <= finish.normalize():

        if is_working_day(current):

            periods = [
                (
                    current + pd.Timedelta(hours=8),
                    current + pd.Timedelta(hours=12)
                ),
                (
                    current + pd.Timedelta(hours=13),
                    current + pd.Timedelta(hours=17)
                )
            ]

            for p_start, p_finish in periods:

                a = max(start, p_start)
                b = min(finish, p_finish)

                if b > a:

                    total += (
                        b - a
                    ).total_seconds() / 3600

        current += pd.Timedelta(days=1)

    return sign * total


# ------------------------------------------------------------
# Red CPM
# ------------------------------------------------------------

network_edges["Lag_Hours"] = pd.to_numeric(
    network_edges["Lag_Hours"],
    errors="coerce"
).fillna(0)

G = nx.DiGraph()

for _, r in network_edges.iterrows():

    G.add_edge(
        r["Predecessor_ID"],
        r["Successor_ID"],
        relationship=r["Relationship_Type"],
        lag_hours=r["Lag_Hours"]
    )


print("✅ Recovery CPM bootstrap ready")
print("TASK:", len(task_xer))
print("Relationships:", G.number_of_edges())
print("Holidays:", len(HOLIDAYS))

In [ ]:
# ============================================================
# 30.2 — CPM parametrizable por Recovery Scenario
# ============================================================

DATA_DATE = (
    pd.Timestamp("2025-08-31")
    + pd.Timedelta(hours=8)
)

cpm_tasks = task_xer[
    [
        "task_code",
        "status_code",
        "remain_drtn_hr_cnt",
        "act_start_date",
        "act_end_date",
        "restart_date"
    ]
].copy()

cpm_tasks = cpm_tasks.set_index(
    "task_code"
)


def run_recovery_cpm(duration_overrides=None):

    if duration_overrides is None:
        duration_overrides = {}

    results = {}

    order = list(
        nx.topological_sort(G)
    )

    for activity in order:

        row = cpm_tasks.loc[activity]

        status = row["status_code"]

        duration = pd.to_numeric(
            row["remain_drtn_hr_cnt"],
            errors="coerce"
        )

        duration = (
            0.0
            if pd.isna(duration)
            else float(duration)
        )

        # Recovery override
        if activity in duration_overrides:

            duration = float(
                duration_overrides[activity]
            )

        # ----------------------------------------
        # COMPLETED
        # ----------------------------------------

        if status == "TK_Complete":

            results[activity] = {
                "Start":
                    pd.Timestamp(
                        row["act_start_date"]
                    ),

                "Finish":
                    pd.Timestamp(
                        row["act_end_date"]
                    )
            }

            continue

        # ----------------------------------------
        # BASE START
        # ----------------------------------------

        if status == "TK_Active":

            if pd.notna(
                row["restart_date"]
            ):

                base_start = pd.Timestamp(
                    row["restart_date"]
                )

            else:

                base_start = DATA_DATE

        else:

            base_start = DATA_DATE

        candidate_starts = [
            base_start
        ]

        incoming = network_edges[
            network_edges["Successor_ID"]
            == activity
        ]

        for _, rel in incoming.iterrows():

            pred = rel[
                "Predecessor_ID"
            ]

            if pred not in results:
                continue

            ps = results[pred]["Start"]
            pf = results[pred]["Finish"]

            relation = rel[
                "Relationship_Type"
            ]

            lag = float(
                rel["Lag_Hours"]
            )

            if relation == "FS":

                c = shift_work_hours(
                    pf,
                    lag
                )

                if lag == 0:
                    c = align_to_work_time(c)

            elif relation == "SS":

                c = shift_work_hours(
                    ps,
                    lag
                )

            elif relation == "FF":

                rf = shift_work_hours(
                    pf,
                    lag
                )

                c = subtract_work_hours(
                    rf,
                    duration
                )

            elif relation == "SF":

                rf = shift_work_hours(
                    ps,
                    lag
                )

                c = subtract_work_hours(
                    rf,
                    duration
                )

            else:
                continue

            candidate_starts.append(c)

        start = max(
            candidate_starts
        )

        if duration > 0:

            start = align_to_work_time(
                start
            )

            finish = add_work_hours(
                start,
                duration
            )

        else:

            finish = start

        results[activity] = {
            "Start": start,
            "Finish": finish
        }

    return results

In [ ]:
# ============================================================
# 30.3 — Exhaustive Recovery Search: 3^4 = 81 scenarios
# ============================================================

activities = [
    "RC01",
    "RC02",
    "RC03",
    "RC04"
]

options_by_activity = {
    a:
    recovery_options[
        recovery_options["Activity_ID"] == a
    ].to_dict("records")
    for a in activities
}


# Current Forecast del motor CPM
base_results = run_recovery_cpm()

BASE_CPM_FINISH = (
    base_results["MILESTONE_FINISH"]["Finish"]
)

print(
    "Current CPM Finish:",
    BASE_CPM_FINISH
)


scenario_rows = []

scenario_number = 0

for combo in product(
    *[
        options_by_activity[a]
        for a in activities
    ]
):

    scenario_number += 1

    overrides = {}

    total_incremental_cost = 0
    total_extra_capacity = 0
    nominal_days_recovered = 0

    levels = {}

    for activity, option in zip(
        activities,
        combo
    ):

        overrides[activity] = (
            option["New_RD_Hours"]
        )

        total_incremental_cost += (
            option["Incremental_Cost"]
        )

        total_extra_capacity += (
            option["Extra_Capacity_Pct"]
        )

        nominal_days_recovered += (
            option["Days_Recovered"]
        )

        levels[activity] = (
            option["Recovery_Level"]
        )


    scenario = run_recovery_cpm(
        overrides
    )

    finish = scenario[
        "MILESTONE_FINISH"
    ]["Finish"]


    # Recuperación efectiva a nivel PROYECTO
    delta_work_hours = (
        working_hours_between(
            finish,
            BASE_CPM_FINISH
        )
    )

    project_days_recovered = (
        delta_work_hours / 8
    )


    scenario_rows.append({

        "Scenario_ID":
            f"S{scenario_number:03d}",

        "RC01_Level":
            levels["RC01"],

        "RC02_Level":
            levels["RC02"],

        "RC03_Level":
            levels["RC03"],

        "RC04_Level":
            levels["RC04"],

        "Project_Finish":
            finish,

        "Project_Days_Recovered":
            project_days_recovered,

        "Nominal_Activity_Days_Recovered":
            nominal_days_recovered,

        "Incremental_Cost":
            total_incremental_cost,

        "Extra_Capacity_Index":
            total_extra_capacity
    })


recovery_scenarios = pd.DataFrame(
    scenario_rows
)


print(
    "Escenarios evaluados:",
    len(recovery_scenarios)
)

print(
    "Máxima recuperación real:",
    recovery_scenarios[
        "Project_Days_Recovered"
    ].max(),
    "días"
)

print(
    "Máxima recuperación nominal:",
    recovery_scenarios[
        "Nominal_Activity_Days_Recovered"
    ].max(),
    "días"
)

In [ ]:
# ============================================================
# 30.4 — Exact Pareto Frontier
# Maximizar días recuperados
# Minimizar costo incremental
# ============================================================

def is_dominated(i, df):

    r = df.loc[i]

    others = df.drop(index=i)

    dominates = (
        (
            others["Project_Days_Recovered"]
            >= r["Project_Days_Recovered"]
        )
        &
        (
            others["Incremental_Cost"]
            <= r["Incremental_Cost"]
        )
        &
        (
            (
                others["Project_Days_Recovered"]
                > r["Project_Days_Recovered"]
            )
            |
            (
                others["Incremental_Cost"]
                < r["Incremental_Cost"]
            )
        )
    )

    return dominates.any()


recovery_scenarios["Pareto"] = [
    not is_dominated(
        i,
        recovery_scenarios
    )
    for i in recovery_scenarios.index
]


pareto_exact = (
    recovery_scenarios[
        recovery_scenarios["Pareto"]
    ]
    .sort_values(
        [
            "Project_Days_Recovered",
            "Incremental_Cost"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# Costo por día REAL recuperado
pareto_exact["Cost_per_Day_Recovered"] = np.where(
    pareto_exact["Project_Days_Recovered"] > 0,

    pareto_exact["Incremental_Cost"]
    /
    pareto_exact["Project_Days_Recovered"],

    np.nan
)


print(
    "Soluciones Pareto exactas:",
    len(pareto_exact)
)

display(
    pareto_exact[
        [
            "Scenario_ID",
            "RC01_Level",
            "RC02_Level",
            "RC03_Level",
            "RC04_Level",
            "Project_Finish",
            "Project_Days_Recovered",
            "Incremental_Cost",
            "Cost_per_Day_Recovered",
            "Extra_Capacity_Index"
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# 31.1 — Baseline Recovery + Marginal Cost Analysis
# ============================================================

BASELINE_FINISH = pd.Timestamp(
    "2026-01-08 17:00"
)

decision_frontier = (
    pareto_exact
    .copy()
    .sort_values(
        "Project_Days_Recovered"
    )
    .reset_index(drop=True)
)

# Variación respecto a baseline en TIEMPO LABORAL
decision_frontier["Finish_Variance_Work_Days"] = (
    decision_frontier["Project_Finish"]
    .apply(
        lambda x:
        working_hours_between(
            BASELINE_FINISH,
            pd.Timestamp(x)
        ) / 8
    )
)

# Incrementos entre puntos Pareto consecutivos
decision_frontier["Marginal_Days_Recovered"] = (
    decision_frontier[
        "Project_Days_Recovered"
    ].diff()
)

decision_frontier["Marginal_Cost"] = (
    decision_frontier[
        "Incremental_Cost"
    ].diff()
)

decision_frontier["Marginal_Cost_per_Day"] = (
    decision_frontier[
        "Marginal_Cost"
    ]
    /
    decision_frontier[
        "Marginal_Days_Recovered"
    ]
)

display(
    decision_frontier[
        [
            "Scenario_ID",
            "Project_Finish",
            "Project_Days_Recovered",
            "Finish_Variance_Work_Days",
            "Incremental_Cost",
            "Marginal_Days_Recovered",
            "Marginal_Cost",
            "Marginal_Cost_per_Day"
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# 31.2 — Minimum-Cost Contractual Recovery
# ============================================================

scenario_decision = recovery_scenarios.copy()

scenario_decision[
    "Finish_Variance_Work_Days"
] = (
    scenario_decision["Project_Finish"]
    .apply(
        lambda x:
        working_hours_between(
            BASELINE_FINISH,
            pd.Timestamp(x)
        ) / 8
    )
)

# Escenarios que llegan al baseline o antes
baseline_recovery_candidates = (
    scenario_decision[
        scenario_decision[
            "Finish_Variance_Work_Days"
        ] <= 0
    ]
    .sort_values(
        "Incremental_Cost"
    )
)

minimum_cost_recovery = (
    baseline_recovery_candidates
    .iloc[0]
)

print("=== MINIMUM COST CONTRACTUAL RECOVERY ===")

display(
    minimum_cost_recovery[
        [
            "Scenario_ID",
            "RC01_Level",
            "RC02_Level",
            "RC03_Level",
            "RC04_Level",
            "Project_Finish",
            "Project_Days_Recovered",
            "Finish_Variance_Work_Days",
            "Incremental_Cost",
            "Extra_Capacity_Index"
        ]
    ].to_frame("Value")
)

## 32 — NSGA-II Recovery Optimization

In [ ]:
# ============================================================
# 32.1 — NSGA-II
# Objectives:
#   F1 = Minimize (-Project Days Recovered)
#   F2 = Minimize Incremental Cost
# ============================================================

import random
from itertools import product
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

GENE_ACTIVITIES = [
    "RC01",
    "RC02",
    "RC03",
    "RC04"
]

GENE_TO_LEVEL = {
    0: "NORMAL",
    1: "MODERATE",
    2: "AGGRESSIVE"
}

LEVEL_TO_GENE = {
    v: k for k, v in GENE_TO_LEVEL.items()
}


# ------------------------------------------------------------
# Scenario lookup
# ------------------------------------------------------------

scenario_lookup = {}

for _, row in recovery_scenarios.iterrows():

    gene = (
        LEVEL_TO_GENE[row["RC01_Level"]],
        LEVEL_TO_GENE[row["RC02_Level"]],
        LEVEL_TO_GENE[row["RC03_Level"]],
        LEVEL_TO_GENE[row["RC04_Level"]]
    )

    scenario_lookup[gene] = row


def evaluate(individual):

    row = scenario_lookup[
        tuple(individual)
    ]

    # NSGA-II minimiza ambos objetivos
    f1 = -float(
        row["Project_Days_Recovered"]
    )

    f2 = float(
        row["Incremental_Cost"]
    )

    return (f1, f2)


def dominates(obj_a, obj_b):

    return (
        obj_a[0] <= obj_b[0]
        and
        obj_a[1] <= obj_b[1]
        and
        (
            obj_a[0] < obj_b[0]
            or
            obj_a[1] < obj_b[1]
        )
    )


# ------------------------------------------------------------
# Fast Non-Dominated Sorting
# ------------------------------------------------------------

def non_dominated_sort(population):

    objectives = [
        evaluate(ind)
        for ind in population
    ]

    S = [
        []
        for _ in population
    ]

    n = [
        0
        for _ in population
    ]

    rank = {}

    fronts = [[]]

    for p in range(len(population)):

        for q in range(len(population)):

            if p == q:
                continue

            if dominates(
                objectives[p],
                objectives[q]
            ):

                S[p].append(q)

            elif dominates(
                objectives[q],
                objectives[p]
            ):

                n[p] += 1

        if n[p] == 0:

            rank[p] = 0
            fronts[0].append(p)

    i = 0

    while (
        i < len(fronts)
        and
        len(fronts[i]) > 0
    ):

        next_front = []

        for p in fronts[i]:

            for q in S[p]:

                n[q] -= 1

                if n[q] == 0:

                    rank[q] = i + 1
                    next_front.append(q)

        if next_front:
            fronts.append(next_front)

        i += 1

    return fronts, rank


# ------------------------------------------------------------
# Crowding Distance
# ------------------------------------------------------------

def crowding_distance(
    population,
    front
):

    distance = {
        i: 0.0
        for i in front
    }

    if len(front) <= 2:

        for i in front:
            distance[i] = np.inf

        return distance

    objectives = {
        i: evaluate(population[i])
        for i in front
    }

    for m in range(2):

        sorted_front = sorted(
            front,
            key=lambda i:
            objectives[i][m]
        )

        distance[
            sorted_front[0]
        ] = np.inf

        distance[
            sorted_front[-1]
        ] = np.inf

        f_min = objectives[
            sorted_front[0]
        ][m]

        f_max = objectives[
            sorted_front[-1]
        ][m]

        if f_max == f_min:
            continue

        for j in range(
            1,
            len(sorted_front) - 1
        ):

            prev_val = objectives[
                sorted_front[j - 1]
            ][m]

            next_val = objectives[
                sorted_front[j + 1]
            ][m]

            distance[
                sorted_front[j]
            ] += (
                next_val - prev_val
            ) / (
                f_max - f_min
            )

    return distance


# ------------------------------------------------------------
# Tournament Selection
# ------------------------------------------------------------

def tournament_selection(
    population,
    ranks,
    crowding
):

    a, b = random.sample(
        range(len(population)),
        2
    )

    if ranks[a] < ranks[b]:
        return population[a]

    if ranks[b] < ranks[a]:
        return population[b]

    ca = crowding.get(
        a,
        0
    )

    cb = crowding.get(
        b,
        0
    )

    if ca > cb:
        return population[a]

    if cb > ca:
        return population[b]

    return random.choice(
        [
            population[a],
            population[b]
        ]
    )


# ------------------------------------------------------------
# Crossover
# ------------------------------------------------------------

def crossover(
    parent1,
    parent2,
    probability=0.90
):

    if random.random() > probability:

        return (
            tuple(parent1),
            tuple(parent2)
        )

    child1 = []
    child2 = []

    for g1, g2 in zip(
        parent1,
        parent2
    ):

        if random.random() < 0.5:

            child1.append(g1)
            child2.append(g2)

        else:

            child1.append(g2)
            child2.append(g1)

    return (
        tuple(child1),
        tuple(child2)
    )


# ------------------------------------------------------------
# Mutation
# ------------------------------------------------------------

def mutate(
    individual,
    probability=0.20
):

    individual = list(
        individual
    )

    for i in range(
        len(individual)
    ):

        if random.random() < probability:

            alternatives = [
                x
                for x in [0, 1, 2]
                if x != individual[i]
            ]

            individual[i] = random.choice(
                alternatives
            )

    return tuple(
        individual
    )


# ------------------------------------------------------------
# NSGA-II parameters
# ------------------------------------------------------------

POP_SIZE = 36
GENERATIONS = 100

ALL_CHROMOSOMES = list(
    product(
        [0, 1, 2],
        repeat=4
    )
)


# Initial random population
population = random.sample(
    ALL_CHROMOSOMES,
    POP_SIZE
)

evaluated_archive = set(
    population
)

history_rows = []


# ------------------------------------------------------------
# Evolution
# ------------------------------------------------------------

for generation in range(
    GENERATIONS
):

    fronts, ranks = (
        non_dominated_sort(
            population
        )
    )

    crowding = {}

    for front in fronts:

        crowding.update(
            crowding_distance(
                population,
                front
            )
        )


    # -----------------------------
    # Create offspring
    # -----------------------------

    offspring = []

    while len(offspring) < POP_SIZE:

        p1 = tournament_selection(
            population,
            ranks,
            crowding
        )

        p2 = tournament_selection(
            population,
            ranks,
            crowding
        )

        c1, c2 = crossover(
            p1,
            p2
        )

        c1 = mutate(c1)
        c2 = mutate(c2)

        offspring.extend(
            [c1, c2]
        )

        evaluated_archive.add(c1)
        evaluated_archive.add(c2)

    offspring = offspring[
        :POP_SIZE
    ]


    # -----------------------------
    # Parent + offspring
    # -----------------------------

    combined = list(
        dict.fromkeys(
            population
            +
            offspring
        )
    )


    # Random immigrants if duplicates
    while (
        len(combined) < POP_SIZE
    ):

        candidate = random.choice(
            ALL_CHROMOSOMES
        )

        if candidate not in combined:

            combined.append(
                candidate
            )

            evaluated_archive.add(
                candidate
            )


    fronts_combined, _ = (
        non_dominated_sort(
            combined
        )
    )

    new_population = []


    for front in fronts_combined:

        if (
            len(new_population)
            +
            len(front)
            <= POP_SIZE
        ):

            new_population.extend(
                [
                    combined[i]
                    for i in front
                ]
            )

        else:

            cd = crowding_distance(
                combined,
                front
            )

            ordered = sorted(
                front,
                key=lambda i:
                cd[i],
                reverse=True
            )

            remaining = (
                POP_SIZE
                -
                len(new_population)
            )

            new_population.extend(
                [
                    combined[i]
                    for i in
                    ordered[:remaining]
                ]
            )

            break

    population = (
        new_population
    )


    # -----------------------------
    # History / convergence
    # -----------------------------

    archive_rows = [
        scenario_lookup[g]
        for g in evaluated_archive
    ]

    archive_temp = pd.DataFrame(
        archive_rows
    )

    contractual = archive_temp[
        archive_temp[
            "Project_Days_Recovered"
        ] >= 9
    ]

    if len(contractual) > 0:

        best_contractual_cost = (
            contractual[
                "Incremental_Cost"
            ].min()
        )

    else:

        best_contractual_cost = np.nan


    history_rows.append({

        "Generation":
            generation + 1,

        "Unique_Scenarios_Evaluated":
            len(evaluated_archive),

        "Max_Project_Days_Recovered":
            archive_temp[
                "Project_Days_Recovered"
            ].max(),

        "Best_Contractual_Recovery_Cost":
            best_contractual_cost
    })


nsga_history = pd.DataFrame(
    history_rows
)


print("✅ NSGA-II completed")
print(
    "Generations:",
    GENERATIONS
)

print(
    "Unique scenarios evaluated:",
    len(evaluated_archive),
    "/ 81"
)

print(
    "Final population:",
    len(population)
)

In [ ]:
# ============================================================
# 32.2 — NSGA-II Pareto Archive
# ============================================================

nsga_archive = pd.DataFrame(
    [
        scenario_lookup[g]
        for g in evaluated_archive
    ]
).drop_duplicates(
    subset=["Scenario_ID"]
).reset_index(drop=True)


nsga_archive["Pareto_NS﻿GA"] = [
    not is_dominated(
        i,
        nsga_archive
    )
    for i in nsga_archive.index
]


nsga_pareto = (
    nsga_archive[
        nsga_archive[
            "Pareto_NS﻿GA"
        ]
    ]
    .sort_values(
        [
            "Project_Days_Recovered",
            "Incremental_Cost"
        ]
    )
    .reset_index(drop=True)
)


nsga_pareto[
    "Cost_per_Day_Recovered"
] = np.where(

    nsga_pareto[
        "Project_Days_Recovered"
    ] > 0,

    nsga_pareto[
        "Incremental_Cost"
    ]
    /
    nsga_pareto[
        "Project_Days_Recovered"
    ],

    np.nan
)


display(
    nsga_pareto[
        [
            "Scenario_ID",
            "RC01_Level",
            "RC02_Level",
            "RC03_Level",
            "RC04_Level",
            "Project_Finish",
            "Project_Days_Recovered",
            "Incremental_Cost",
            "Cost_per_Day_Recovered"
        ]
    ].round(2)
)

In [ ]:
# ============================================================
# 32.3 — Validation:
# NSGA-II vs Exact Exhaustive Pareto
# ============================================================

exact_ids = set(
    pareto_exact[
        "Scenario_ID"
    ]
)

nsga_ids = set(
    nsga_pareto[
        "Scenario_ID"
    ]
)

matched_ids = (
    exact_ids
    &
    nsga_ids
)

missing_ids = (
    exact_ids
    -
    nsga_ids
)

extra_ids = (
    nsga_ids
    -
    exact_ids
)


validation_summary = pd.DataFrame({

    "Metric": [
        "Total Search Space",
        "Scenarios Evaluated by NSGA-II",
        "Exact Pareto Solutions",
        "Pareto Solutions Found by NSGA-II",
        "Exact Pareto Solutions Recovered",
        "Validation Accuracy"
    ],

    "Value": [
        81,
        len(evaluated_archive),
        len(exact_ids),
        len(nsga_ids),
        len(matched_ids),
        (
            len(matched_ids)
            /
            len(exact_ids)
        )
    ]
})


display(
    validation_summary
)

print(
    "Exact Pareto:",
    sorted(exact_ids)
)

print(
    "NSGA-II Pareto:",
    sorted(nsga_ids)
)

print(
    "Missing:",
    sorted(missing_ids)
)

print(
    "Extra:",
    sorted(extra_ids)
)


if (
    len(missing_ids) == 0
    and
    len(extra_ids) == 0
):

    print(
        "\n✅ NSGA-II VALIDATION: PASS"
    )

    print(
        "The evolutionary algorithm reproduced "
        "the exact Pareto frontier."
    )

else:

    print(
        "\n⚠️ NSGA-II VALIDATION: PARTIAL"
    )

In [ ]:
# ============================================================
# 33 — Optimization Checkpoint
# ============================================================

# Guardar escenarios completos
recovery_scenarios.to_csv(
    CURATED_DIR / "recovery_scenarios.csv",
    index=False
)

# Frontera exacta
pareto_exact.to_csv(
    CURATED_DIR / "pareto_exact.csv",
    index=False
)

# Frontera encontrada por NSGA-II
nsga_pareto.to_csv(
    CURATED_DIR / "nsga_pareto.csv",
    index=False
)

# Historial de evolución
nsga_history.to_csv(
    CURATED_DIR / "nsga_history.csv",
    index=False
)

# Validación
validation_summary.to_csv(
    CURATED_DIR / "nsga_validation_summary.csv",
    index=False
)

# Análisis de decisión
decision_frontier.to_csv(
    CURATED_DIR / "decision_frontier.csv",
    index=False
)

# Escenario contractual seleccionado
pd.DataFrame(
    [minimum_cost_recovery]
).to_csv(
    CURATED_DIR / "recommended_recovery_scenario.csv",
    index=False
)

print("=== OPTIMIZATION CHECKPOINT ===")
print(
    "Recovery scenarios:",
    len(recovery_scenarios)
)
print(
    "Exact Pareto:",
    len(pareto_exact)
)
print(
    "NSGA-II Pareto:",
    len(nsga_pareto)
)
print(
    "Recommended scenario:",
    minimum_cost_recovery["Scenario_ID"]
)
print(
    "Recommended incremental cost: S/",
    round(
        minimum_cost_recovery["Incremental_Cost"],
        2
    )
)

In [ ]:
# ============================================================
# 34.1 — POWER BI DATA MART
# Dimensions
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

POWERBI_DIR = OUTPUT_DIR / "powerbi"
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

# Recuperar tablas si el runtime no las tiene
if "fact_activity_snapshot" not in globals():
    fact_activity_snapshot = pd.read_csv(
        CURATED_DIR / "fact_activity_snapshot.csv",
        low_memory=False
    )

if "fact_resource_snapshot" not in globals():
    fact_resource_snapshot = pd.read_csv(
        CURATED_DIR / "fact_resource_snapshot.csv",
        low_memory=False
    )

# ------------------------------------------------------------
# DIM ACTIVITY
# ------------------------------------------------------------

activity_cols = [
    c for c in [
        "Activity_ID",
        "Activity_Name",
        "WBS",
        "WBS_Name"
    ]
    if c in fact_evm.columns
]

dim_activity = (
    fact_evm[activity_cols]
    .drop_duplicates(subset=["Activity_ID"])
    .sort_values("Activity_ID")
    .reset_index(drop=True)
)

# Añadir información de criticidad
crit_cols = [
    c for c in [
        "Activity_ID",
        "Critical_Path_Count",
        "Critical_Path_Pct"
    ]
    if c in critical_node_summary.columns
]

if len(crit_cols) > 1:
    dim_activity = dim_activity.merge(
        critical_node_summary[crit_cols]
        .drop_duplicates("Activity_ID"),
        on="Activity_ID",
        how="left"
    )

dim_activity["Is_Recovery_Candidate"] = (
    dim_activity["Activity_ID"]
    .isin(["RC01", "RC02", "RC03", "RC04"])
)

dim_activity["Critical_Path_Pct"] = (
    dim_activity
    .get("Critical_Path_Pct", 0)
    .fillna(0)
)

# ------------------------------------------------------------
# DIM RESOURCE
# ------------------------------------------------------------

resource_cols = [
    c for c in [
        "Resource_ID",
        "Recurso",
        "Unidad",
        "Precio_unit"
    ]
    if c in fact_resource_snapshot.columns
]

dim_resource = (
    fact_resource_snapshot[resource_cols]
    .drop_duplicates(subset=["Resource_ID"])
    .sort_values("Resource_ID")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# DIM CUT
# ------------------------------------------------------------

cut_dates = {
    "C01": "2025-05-31",
    "C02": "2025-06-30",
    "C03": "2025-07-31",
    "C04": "2025-08-31"
}

cut_names = {
    "C01": "Mayo 2026",
    "C02": "Junio 2026",
    "C03": "Julio 2026",
    "C04": "Agosto 2026"
}

dim_cut = pd.DataFrame({
    "Cut_ID": list(cut_dates.keys()),
    "Snapshot_Date": pd.to_datetime(
        list(cut_dates.values())
    )
})

dim_cut["Cut_Name"] = (
    dim_cut["Cut_ID"].map(cut_names)
)

dim_cut["Cut_Order"] = range(
    1,
    len(dim_cut) + 1
)

# ------------------------------------------------------------
# DIM DATE
# ------------------------------------------------------------

date_start = pd.Timestamp("2025-05-01")
date_end   = pd.Timestamp("2026-02-28")

dim_date = pd.DataFrame({
    "Date": pd.date_range(
        date_start,
        date_end,
        freq="D"
    )
})

dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month_No"] = dim_date["Date"].dt.month
dim_date["Month"] = dim_date["Date"].dt.strftime("%B")
dim_date["Year_Month"] = dim_date["Date"].dt.strftime("%Y-%m")
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["Weekday_No"] = dim_date["Date"].dt.weekday + 1
dim_date["Weekday"] = dim_date["Date"].dt.strftime("%A")

print("✅ Dimensions created")
print("Activities:", len(dim_activity))
print("Resources:", len(dim_resource))
print("Cuts:", len(dim_cut))
print("Dates:", len(dim_date))

In [ ]:
# ============================================================
# 34.2 — POWER BI FACT TABLES
# ============================================================

# ------------------------------------------------------------
# Activity Snapshot
# ------------------------------------------------------------

fact_activity_pbi = fact_activity_snapshot.copy()

# ------------------------------------------------------------
# Resource Snapshot
# ------------------------------------------------------------

fact_resource_pbi = fact_resource_snapshot.copy()

# ------------------------------------------------------------
# EVM Activity-Level
# ------------------------------------------------------------

fact_evm_pbi = fact_evm.copy()

# ------------------------------------------------------------
# Project Controls Scorecard
# 1 registro por corte
# ------------------------------------------------------------

fact_project_snapshot = (
    project_controls_scorecard
    .copy()
)

# ------------------------------------------------------------
# Trends
# ------------------------------------------------------------

fact_trends = trend_register.copy()

# ------------------------------------------------------------
# Forecast
# ------------------------------------------------------------

fact_forecast = forecast_summary.copy()

# ------------------------------------------------------------
# Recovery — all scenarios
# ------------------------------------------------------------

fact_recovery_scenarios = (
    recovery_scenarios.copy()
)

# ------------------------------------------------------------
# Pareto frontier
# ------------------------------------------------------------

fact_recovery_pareto = (
    pareto_exact.copy()
)

# Etiquetas gerenciales
decision_labels = {
    "S001": "NO RECOVERY",
    "S021": "CONTRACTUAL RECOVERY",
    "S027": "ENHANCED RECOVERY",
    "S054": "MAXIMUM RECOVERY"
}

fact_recovery_pareto["Decision_Label"] = (
    fact_recovery_pareto["Scenario_ID"]
    .map(decision_labels)
    .fillna("PARETO ALTERNATIVE")
)

# ------------------------------------------------------------
# Recommended recovery
# ------------------------------------------------------------

fact_recovery_recommended = pd.DataFrame(
    [minimum_cost_recovery]
)

fact_recovery_recommended[
    "Recommendation"
] = "MINIMUM COST CONTRACTUAL RECOVERY"

# ------------------------------------------------------------
# Critical Node Analysis
# ------------------------------------------------------------

fact_critical_nodes = (
    critical_node_summary.copy()
)

# ------------------------------------------------------------
# NSGA-II convergence
# ------------------------------------------------------------

fact_nsga_history = (
    nsga_history.copy()
)

# ------------------------------------------------------------
# NSGA-II validation
# ------------------------------------------------------------

fact_nsga_validation = (
    validation_summary.copy()
)

print("✅ Power BI fact tables created")

In [ ]:
# ============================================================
# 34.3 — EXPORT POWER BI DATA MART
# ============================================================

powerbi_tables = {

    # Dimensions
    "dim_activity":
        dim_activity,

    "dim_resource":
        dim_resource,

    "dim_cut":
        dim_cut,

    "dim_date":
        dim_date,

    # Project Controls
    "fact_activity_snapshot":
        fact_activity_pbi,

    "fact_resource_snapshot":
        fact_resource_pbi,

    "fact_evm":
        fact_evm_pbi,

    "fact_project_snapshot":
        fact_project_snapshot,

    "fact_trends":
        fact_trends,

    "fact_forecast":
        fact_forecast,

    "fact_critical_nodes":
        fact_critical_nodes,

    # Recovery / Optimization
    "fact_recovery_scenarios":
        fact_recovery_scenarios,

    "fact_recovery_pareto":
        fact_recovery_pareto,

    "fact_recovery_recommended":
        fact_recovery_recommended,

    "fact_nsga_history":
        fact_nsga_history,

    "fact_nsga_validation":
        fact_nsga_validation
}


manifest_rows = []

for table_name, df in powerbi_tables.items():

    file_path = (
        POWERBI_DIR /
        f"{table_name}.csv"
    )

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d %H:%M:%S"
    )

    manifest_rows.append({
        "Table_Name":
            table_name,

        "Rows":
            len(df),

        "Columns":
            len(df.columns),

        "File":
            file_path.name
    })


powerbi_manifest = pd.DataFrame(
    manifest_rows
)

powerbi_manifest.to_csv(
    POWERBI_DIR / "powerbi_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

display(powerbi_manifest)

print(
    "\n✅ Power BI package exported to:"
)

print(POWERBI_DIR)

In [ ]:
# ============================================================
# 34.4 — POWER BI DATA MART QA
# ============================================================

qa_rows = []

def add_qa(test, result, detail):
    qa_rows.append({
        "Test": test,
        "Result": "PASS" if result else "FAIL",
        "Detail": detail
    })


add_qa(
    "dim_activity unique key",
    dim_activity["Activity_ID"].is_unique,
    f"{dim_activity['Activity_ID'].nunique()} unique IDs"
)

add_qa(
    "dim_resource unique key",
    dim_resource["Resource_ID"].is_unique,
    f"{dim_resource['Resource_ID'].nunique()} unique IDs"
)

add_qa(
    "dim_cut unique key",
    dim_cut["Cut_ID"].is_unique,
    f"{dim_cut['Cut_ID'].nunique()} cuts"
)

add_qa(
    "fact_evm composite key",
    not fact_evm_pbi.duplicated(
        ["Cut_ID", "Activity_ID"]
    ).any(),
    "Cut_ID + Activity_ID"
)

add_qa(
    "fact_resource composite key",
    not fact_resource_pbi.duplicated(
        ["Cut_ID", "Activity_ID", "Resource_ID"]
    ).any(),
    "Cut_ID + Activity_ID + Resource_ID"
)

add_qa(
    "Recovery scenarios",
    len(fact_recovery_scenarios) == 81,
    f"{len(fact_recovery_scenarios)} scenarios"
)

add_qa(
    "Exact Pareto",
    len(fact_recovery_pareto) == 10,
    f"{len(fact_recovery_pareto)} Pareto solutions"
)

add_qa(
    "Recommended scenario",
    fact_recovery_recommended[
        "Scenario_ID"
    ].iloc[0] == "S021",
    fact_recovery_recommended[
        "Scenario_ID"
    ].iloc[0]
)

qa_powerbi = pd.DataFrame(
    qa_rows
)

display(qa_powerbi)

print(
    "\nOverall:",
    "PASS"
    if (qa_powerbi["Result"] == "PASS").all()
    else "CHECK REQUIRED"
)

In [ ]:
# ============================================================
# 34.5 — Enrich dim_activity with P6 Baseline Dates
# ============================================================

baseline_file = RAW_DIR / "01_Activities.xlsx"

baseline_raw = pd.read_excel(baseline_file)

print(baseline_raw.columns.tolist())
display(baseline_raw.head())

In [ ]:
# ============================================================
# RESUME PATHS
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_Controls_Recovery_Demo"
)

RAW_DIR = PROJECT_ROOT / "01_raw"
CURATED_DIR = PROJECT_ROOT / "02_curated"
OUTPUT_DIR = PROJECT_ROOT / "03_outputs"
AUDIT_DIR = PROJECT_ROOT / "04_audit"
POWERBI_DIR = OUTPUT_DIR / "powerbi"

print("RAW_DIR:", RAW_DIR)
print("Exists:", RAW_DIR.exists())

In [ ]:
# ============================================================
# 34.5 — Enrich dim_activity with frozen P6 baseline dates
# ============================================================

# Recuperar dim_activity desde el Data Mart persistido
dim_activity = pd.read_csv(
    POWERBI_DIR / "dim_activity.csv",
    low_memory=False
)

# Limpiar baseline P6
baseline_pbi = (
    baseline_raw[
        baseline_raw["task_code"] != "Activity ID"
    ][
        [
            "task_code",
            "start_date",
            "end_date"
        ]
    ]
    .rename(
        columns={
            "task_code": "Activity_ID",
            "start_date": "Baseline_Start",
            "end_date": "Baseline_Finish"
        }
    )
    .copy()
)

# Limpiar IDs
baseline_pbi["Activity_ID"] = (
    baseline_pbi["Activity_ID"]
    .astype(str)
    .str.strip()
)

# Tipos fecha
baseline_pbi["Baseline_Start"] = pd.to_datetime(
    baseline_pbi["Baseline_Start"],
    errors="coerce"
)

baseline_pbi["Baseline_Finish"] = pd.to_datetime(
    baseline_pbi["Baseline_Finish"],
    errors="coerce"
)

# Evitar duplicar columnas si volvemos a ejecutar la celda
dim_activity = dim_activity.drop(
    columns=[
        "Baseline_Start",
        "Baseline_Finish"
    ],
    errors="ignore"
)

# Merge por Activity ID
dim_activity = dim_activity.merge(
    baseline_pbi,
    on="Activity_ID",
    how="left",
    validate="one_to_one"
)

print("=== BASELINE ENRICHMENT QA ===")
print("Activities dim_activity:", len(dim_activity))
print("Activities baseline:", len(baseline_pbi))
print(
    "Baseline Start missing:",
    dim_activity["Baseline_Start"].isna().sum()
)
print(
    "Baseline Finish missing:",
    dim_activity["Baseline_Finish"].isna().sum()
)

display(
    dim_activity[
        [
            "Activity_ID",
            "Activity_Name",
            "Baseline_Start",
            "Baseline_Finish"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# QA — Baseline activities not included in dim_activity
# ============================================================

extra_baseline = (
    baseline_pbi[
        ~baseline_pbi["Activity_ID"].isin(
            dim_activity["Activity_ID"]
        )
    ]
)

display(extra_baseline)

print(
    "Baseline IDs fuera de dim_activity:",
    extra_baseline["Activity_ID"].tolist()
)

In [ ]:
# ============================================================
# 34.6 — Persist updated Power BI dimension
# ============================================================

dim_activity.to_csv(
    POWERBI_DIR / "dim_activity.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d %H:%M:%S"
)

print(
    "✅ dim_activity.csv actualizado con "
    "Baseline_Start y Baseline_Finish"
)